# GRANDE-U — Reproducible groundwater-level reconstruction
**A Two-Step Machine-Learning Framework for Reconstructing Groundwater-Level Time Series in Transboundary Aquifers**

[Open on GitHub](https://github.com/VytautasSam/grande-u-gwl-reconstruction) · [Open in Google Colab](https://colab.research.google.com/github/VytautasSam/grande-u-gwl-reconstruction/blob/main/GRANDE_U_manuscript_reproducible_workflow.ipynb)

This notebook runs, from top to bottom: **software setup → data source loading + checksum verification → harmonization/QC → exploratory statistics → feature engineering → chronologically evaluated Step-1 and Step-2 per-well models → SHAP → PCA/K-means clustering → Figures 1–9 + supporting tables, in manuscript order.**

### How to run
1. **Colab:** Open the link above (once this repository is published) and use **Runtime → Run all**. Input files will be reconstructed from small lossless chunks archived in this repository, and SHA-256 verified, without manual uploads. A public repository is required for anonymous downloads; see README for private repositories.
2. **Local:** Clone/download the repository, install `requirements.txt`, open this notebook from the repository root and run all cells. The `data_inputs/` files are already included.
3. **Outputs:** Saved under `manuscript_outputs/figures/` and `manuscript_outputs/tables/` (these derived artifacts are Git-ignored). For long analyses prefer a Colab high-memory runtime; full per-well ML and SHAP may be expensive.

**Scientific reproducibility:** Input SHA-256 hashes are checked against files supplied for this repository. Random seed 42, model settings, time-series splits, reference-well selection, and other methodological choices are retained from the combined notebook. This is **not** a re-creation of the upstream national data acquisition, ERA5 extraction, or initial GRACE imputation. See `data_inputs/README.md` and the final audit notes.

**Validation:** The supplied combined notebook was reported to run successfully. This GitHub adaptation changes installation and data retrieval only; a full manuscript-scale numerical run of the adapted notebook has not been verified in this environment.


## 0. Input inventory and provenance
All four inputs are included in this repository. Three are stored losslessly as small compressed chunks under `data_inputs/packed/`; the static Excel file remains directly accessible. The exact file names and SHA-256 hashes are also recorded in `data_inputs/manifest.json` and checked at runtime. **All external inputs are parsed in the next source-loading cell.**

| Input alias | Repository file | Earlier notebook / source | What it provides |
|---|---|---|---|
| `monthly_wells` | `data_inputs/monthly_wells.csv` | `1_data_processing_and_validation`, cell 3 | Harmonized GWL time series, coordinates and integrated hydroclimate variables |
| `static_wells` | `data_inputs/static_wells.xlsx` | `1_data_processing_and_validation`, cell 5 | Well depth, altitude, lithology, karst, confinement, pumping and aquifer properties |
| `processed_grace` | `data_inputs/processed_grace.parquet` | `1_data_processing_and_validation`, cell 11 | Already-infilled GRACE/GRACE-FO TWS data, associated spatial polygons |
| `world_boundaries` | `data_inputs/world_boundaries.geojson` | `2_statistics_analytics_maps`, cell 15 | Country boundaries used in Figure 1 |

### Upstream provenance (manuscript, §3.1)
Groundwater: Estonia **EELIS/KESE**; Latvia **LEGMC**; Lithuania **PožVIS/LGS**; Poland **PGI–NRI**; United States **NGWMN/USGS**; Ukraine **Zakhidukrgeologiya, Geoinform of Ukraine, Lviv Hydrogeological and Reclamation Expedition**, and additional operators. Climatic forcing: **ERA5-Land**, with AET and PET present in the input table. Regional TWS: **GRACE/GRACE-FO**. The original source URLs and a description of these integrated products are retained in the JSON manifest.

**Scope:** The repository contains the integrated/processed four input files, not the national agencies’ raw feeds or the upstream satellite imputation workflow. Verify redistributability of the source datasets before publishing the repository publicly.


In [ ]:
# Runtime bootstrap: safe to execute in Colab or an ordinary Jupyter kernel.
# Colab opens .ipynb files from GitHub without cloning the repository;
# the input-loader below retrieves small chunks and restores the four version-checked files as needed.
import importlib.util
import subprocess
import sys

# Install only missing modules; most are included in Colab already.
DEPENDENCIES = {
    'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<3',
    'scipy': 'scipy>=1.12,<2', 'matplotlib': 'matplotlib>=3.8,<4',
    'geopandas': 'geopandas>=0.14,<2', 'shapely': 'shapely>=2,<3',
    'pyarrow': 'pyarrow>=16,<25', 'openpyxl': 'openpyxl>=3.1,<4',
    'requests': 'requests>=2.31,<3', 'sklearn': 'scikit-learn>=1.4,<2',
    'shap': 'shap>=0.45,<1',
}
missing = [spec for module, spec in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing dependencies:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import hashlib, json, math, os, platform, re, string, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, FancyBboxPatch
from matplotlib.colors import Normalize
from shapely.geometry import Point
from shapely import wkb
import requests
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
import shap

SEED = 42
np.random.seed(SEED)
INPUT_DIR = Path('data_inputs'); INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('manuscript_outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUTPUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = OUTPUT_DIR / 'tables'; TABLE_DIR.mkdir(exist_ok=True)
DPI = 600
AREA_ORDER = ['LT-LV transboundary', 'LV-EE transboundary', 'PL-UA transboundary', 'US-MX transboundary']
AREA_SHORT = {x: x.split(' ')[0].replace('-', '–') for x in AREA_ORDER}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': False, 'savefig.facecolor': 'white'})


In [ ]:
import shutil
# Centralized, immutable source registry (original URLs and exact supplied file hashes).
# This cell is self-contained: in Colab the repository data files are downloaded
# from GitHub, while local execution uses the checked-in data_inputs/ files.
REPO_RAW = 'https://raw.githubusercontent.com/VytautasSam/grande-u-gwl-reconstruction/main/data_inputs'
SOURCES = {
    'monthly_wells': {'url': 'https://drive.google.com/file/d/1hGvpwOjsTCA3RHDdohkN4wpaXlI6QIGQ/view?usp=drive_link', 'path': INPUT_DIR/'monthly_wells.csv', 'format': 'csv',
         'origin': '1_data_processing_and_validation notebook, cell 3', 'sha256': '116153a17b115a166c68b385263aa73e8c28c6ee6c014d3c26e748ff930982a3',
         'storage': 'gzip_parts'},
    'static_wells': {'url': 'https://docs.google.com/spreadsheets/d/1HtP20Kzj6cMaFUL46Kq8jB7pQ-VKzRx9/edit', 'path': INPUT_DIR/'static_wells.xlsx', 'format': 'xlsx',
         'origin': '1_data_processing_and_validation notebook, cell 5', 'sha256': '9585262aebb0c5882133b88cfec1fc5d12a898a7d29e837161099aefaf561772',
         'repo_url': REPO_RAW + '/static_wells.xlsx'},
    'processed_grace': {'url': 'https://drive.google.com/file/d/1y0imJalS9fuwDZtdEweSHBa0ZWFXgCn2/view?usp=drive_link', 'path': INPUT_DIR/'processed_grace.parquet', 'format': 'parquet',
         'origin': '1_data_processing_and_validation notebook, cell 11', 'sha256': '499699ce5e837a048ff83da0f9d26138f394afad9b68cc84f744f968eb3cb33e',
         'storage': 'gzip_parts'},
    'world_boundaries': {'url': 'https://drive.google.com/file/d/13h_dMZ46UwB4Q1Eyh9fYYS2xhJHUn-P_/view', 'path': INPUT_DIR/'world_boundaries.geojson', 'format': 'geojson',
         'origin': '2_statistics_analytics_maps notebook, cell 15', 'sha256': '764f259082c141debd112178117f2a76fe40c585415b1edd1466a43d20d15fea',
         'storage': 'gzip_parts'},
}

# Exact packed archive metadata from data_inputs/manifest.json; small files upload cleanly to GitHub.
PACKED_PARTS = {'monthly_wells': [{'file': 'packed/monthly_wells.csv.gz.part000', 'bytes': 768000, 'sha256': '108c7222fadda1e8d58b078a02c4e57a06163299d2529aa3fbf8bcd9f3b23096'}, {'file': 'packed/monthly_wells.csv.gz.part001', 'bytes': 768000, 'sha256': '0a9d193d4b4269ef55ebecd043fbb634c9fcd182861539c4e85ba403cb24a032'}, {'file': 'packed/monthly_wells.csv.gz.part002', 'bytes': 768000, 'sha256': '129adbb545d48d6da5dec0df29ec1b9373d238ad203ff2abae86aacb6a23c8e8'}, {'file': 'packed/monthly_wells.csv.gz.part003', 'bytes': 768000, 'sha256': '87d173e549125c551d962158835eec60e687fd65cdff58d080fe4776089f3c8d'}, {'file': 'packed/monthly_wells.csv.gz.part004', 'bytes': 768000, 'sha256': '1dc2f732b05fb2f74a7fe4df55e3c2b4249739f29c0b6457b1a6646092317a21'}, {'file': 'packed/monthly_wells.csv.gz.part005', 'bytes': 768000, 'sha256': '7fdc84f480b7fb8a2c7c5eacecfb2faf0fe316f6d03af3ccd74b8b9f09de930d'}, {'file': 'packed/monthly_wells.csv.gz.part006', 'bytes': 516087, 'sha256': '298babbc3e4b50a4c7f68e62da8df32c0c51209fe42940494d053b4898046d50'}], 'processed_grace': [{'file': 'packed/processed_grace.parquet.gz.part000', 'bytes': 768000, 'sha256': '60a5581a4b84ed587032224ec1d0be82b8ffad1907f7ad6afdc687534e3eae06'}, {'file': 'packed/processed_grace.parquet.gz.part001', 'bytes': 768000, 'sha256': 'da84746c29db162151ccfb6c7b5f5e30ff7b578472a9ba7eb0c2baecf6dc39d5'}, {'file': 'packed/processed_grace.parquet.gz.part002', 'bytes': 768000, 'sha256': '4467a02eb7c4f17af9d47ada977940eb43493ad8765cd0046ef200a677b303d1'}, {'file': 'packed/processed_grace.parquet.gz.part003', 'bytes': 493013, 'sha256': '0dd3bce42722e0d23e81ef968966afd9e4028266434de76b39d2ab1ce981e259'}], 'world_boundaries': [{'file': 'packed/world_boundaries.geojson.gz.part000', 'bytes': 768000, 'sha256': 'df8826c353ec64a5fd547919251da81fe74ab4da99f7628a5ee8c9ab3c39ab62'}, {'file': 'packed/world_boundaries.geojson.gz.part001', 'bytes': 768000, 'sha256': '11fa981cd0161af2d0b1a2bbe2ded93930eae5bd0f668a713acd932c90da5d21'}, {'file': 'packed/world_boundaries.geojson.gz.part002', 'bytes': 768000, 'sha256': '0857ca51e3e9f09a2aa72c62677ff9523e8d381c8b98319a5e0db5128b5718b9'}, {'file': 'packed/world_boundaries.geojson.gz.part003', 'bytes': 768000, 'sha256': '036909abf688d2f9a0f2bf9b7f247d06ca2683faee6c04b3c3a2a7d091ed0566'}, {'file': 'packed/world_boundaries.geojson.gz.part004', 'bytes': 768000, 'sha256': 'ad7484d3eaabe711df550eb5a0ca2235f731cf9c0ccc6b41b501c67bcab08df0'}, {'file': 'packed/world_boundaries.geojson.gz.part005', 'bytes': 768000, 'sha256': '05cdc583f550e393c31e5f05c124381a89e5a8e2a27ee986d02ee5fe53ac7462'}, {'file': 'packed/world_boundaries.geojson.gz.part006', 'bytes': 768000, 'sha256': '6e134e012e9eccaa3bcd49002bd137a3c5aedaa5a7a83929b31ce71c1984c3bc'}, {'file': 'packed/world_boundaries.geojson.gz.part007', 'bytes': 768000, 'sha256': '572f31e77c63053b72aa76fe98ecf6715aad1c72bf1e96f91daf9d47fbfb65ae'}, {'file': 'packed/world_boundaries.geojson.gz.part008', 'bytes': 768000, 'sha256': '72365ffde116b936001134f0c388b0599262b3012b35ef6d1572bc558827a03d'}, {'file': 'packed/world_boundaries.geojson.gz.part009', 'bytes': 605279, 'sha256': '6112b7c7debf7f23e6d9b720618dc7bbae91d8e8d163c25e4393be521bd31139'}]}

def direct_url(url):
    m = re.search(r'/d/([\w-]+)', url)
    if not m:
        raise ValueError(f'Unrecognized Google Drive link: {url}')
    file_id = m.group(1)
    if 'docs.google.com/spreadsheets' in url:
        return f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx'
    return f'https://drive.google.com/uc?export=download&id={file_id}'

def download_verified(url, dest, expected_sha256):
    """Download one repository file and verify its exact SHA-256 digest."""
    tmp = dest.with_name(dest.name + '.download')
    try:
        with requests.get(url, timeout=180, stream=True) as response:
            response.raise_for_status()
            if 'text/html' in response.headers.get('content-type', '').lower():
                raise RuntimeError('Received HTML instead of a data file')
            with tmp.open('wb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        actual = digest(tmp)
        if actual != expected_sha256:
            raise RuntimeError(f'SHA-256 mismatch: expected {expected_sha256}, got {actual}')
        tmp.replace(dest)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def cached_input(alias):
    """Restore exact archived input from small compressed repository parts.

    The repository stores large files in chunks so GitHub web uploads remain
    manageable. Existing originals are accepted ONLY if their SHA-256 matches.
    Works in a local clone or directly in Colab without cloning the repo.
    """
    spec = SOURCES[alias]
    path = Path(spec['path'])
    if path.is_file():
        if digest(path) != spec['sha256']:
            raise RuntimeError(f'Invalid checksum for existing input {path}; please remove the incorrect file.')
        return path

    if alias == 'static_wells':
        download_verified(spec['repo_url'], path, spec['sha256'])
        return path

    # Download each compressed part only when absent; verify every part.
    # The checked-in manifest is embedded below to avoid a second remote fetch.
    parts = PACKED_PARTS[alias]
    packed_dir = INPUT_DIR / 'packed'
    packed_dir.mkdir(parents=True, exist_ok=True)
    for part in parts:
        part_path = INPUT_DIR / part['file']
        if part_path.exists():
            if digest(part_path) != part['sha256']:
                raise RuntimeError(f'Invalid checksum for archived part {part_path}.')
        else:
            download_verified(REPO_RAW + '/' + part['file'], part_path, part['sha256'])
        if part_path.stat().st_size != part['bytes']:
            raise RuntimeError(f'Wrong size for {part_path}.')

    # Join the gzip members as bytes (these are chunks of ONE gzip stream),
    # decompress losslessly, and validate the original dataset checksum.
    import gzip
    compressed = INPUT_DIR / (path.name + '.assembled.gz')
    restored = INPUT_DIR / (path.name + '.restore-tmp')
    try:
        with compressed.open('wb') as output:
            for part in parts:
                with (INPUT_DIR / part['file']).open('rb') as f:
                    shutil.copyfileobj(f, output, 1024 * 1024)
        with gzip.open(compressed, 'rb') as source, restored.open('wb') as output:
            shutil.copyfileobj(source, output, 1024 * 1024)
        if digest(restored) != spec['sha256']:
            raise RuntimeError(f'Reconstructed {alias} does not match the original source SHA-256.')
        restored.replace(path)
    finally:
        restored.unlink(missing_ok=True)
        compressed.unlink(missing_ok=True)
    print(f'Restored & verified {alias}: {path.stat().st_size:,} bytes from {len(parts)} archived chunks')
    return path


def load_input(alias):
    path = cached_input(alias); fmt = SOURCES[alias]['format']
    if fmt == 'csv': return pd.read_csv(path, low_memory=False)
    if fmt == 'xlsx': return pd.read_excel(path)
    if fmt == 'parquet': return pd.read_parquet(path)
    if fmt == 'geojson': return gpd.read_file(path)
    raise ValueError(fmt)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20), b''): h.update(chunk)
    return h.hexdigest()

# ALL EXTERNAL SOURCE FILES ARE READ HERE, AT THE BEGINNING. There are no
# later Drive-download calls and no dependence on the separate plotting exports.
df_wells = load_input('monthly_wells')
df_static = load_input('static_wells')
df_grace_imputed = load_input('processed_grace')
world = load_input('world_boundaries')

source_manifest = pd.DataFrame([
    {'alias': alias, 'original_url': spec['url'], 'source_cell': spec['origin'],
     'cached_file': str(spec['path']), 'bytes': Path(spec['path']).stat().st_size,
     'sha256': digest(spec['path'])} for alias, spec in SOURCES.items()
])
source_manifest.to_csv(TABLE_DIR/'00_source_manifest.csv', index=False)
display(source_manifest[['alias','source_cell','bytes','sha256']])
print('Input row counts:', {name: len(obj) for name,obj in
      [('monthly wells',df_wells),('static wells',df_static),
       ('GRACE rows',df_grace_imputed),('world polygons',world)]})

## 1. Harmonize sources and validate join structure
Retained from data-processing notebook: WGS84 well points; GRACE GeoParquet geometry recovery; **point-within-polygon** match within `year_month`; one polygon per source record if overlapping; period filter **2003-01-01 through 2023-12-23**; left join on `well_ID`; national and transboundary assignment (Latvian latitude split at 57°N). Source-level duplicate and unmatched-data diagnostics are exported.

Do **not** silently deduplicate well-month records: the model assumes a chronological well-month table, and row-based lagging can be wrong if there are missing or duplicate months. Diagnostics here make that visible.

In [ ]:
STATIC_COLS = ['well_ID','aquifer_thickness','aquifer_lithology','aquifer_karst',
               'confinement','ground_altitude','well_depth','pumping','avg_gw_abs']
missing_wells = {'well_ID','date','Long_WGS84','Lat_WGS84','gw_level_abs'}-set(df_wells)
missing_static = set(STATIC_COLS)-set(df_static)
missing_grace = {'year_month','geometry','lwe_imputed'}-set(df_grace_imputed)
if missing_wells or missing_static or missing_grace:
    raise KeyError(f'Input schema missing: well={missing_wells}; static={missing_static}; GRACE={missing_grace}')

def as_geometry(frame):
    frame=frame.copy()
    # Grace polygons in the supplied parquet were stored as WKB bytes.
    if not isinstance(frame, gpd.GeoDataFrame):
        vals=frame['geometry']
        if len(vals) and isinstance(vals.dropna().iloc[0], (bytes,bytearray,memoryview)):
            geo=gpd.GeoSeries.from_wkb(vals)
        else:
            geo=gpd.GeoSeries(vals)
        return gpd.GeoDataFrame(frame.drop(columns='geometry'), geometry=geo, crs='EPSG:4326')
    return frame.set_crs('EPSG:4326',allow_override=True) if frame.crs is None else frame

gdf_wells=gpd.GeoDataFrame(df_wells.copy(), geometry=gpd.points_from_xy(
    df_wells['Long_WGS84'],df_wells['Lat_WGS84']), crs='EPSG:4326')
gdf_wells['date']=pd.to_datetime(gdf_wells['date'],errors='coerce')
gdf_wells['year_month']=gdf_wells['date'].dt.strftime('%Y-%m')
gdf_grace_imputed=as_geometry(df_grace_imputed)
gdf_grace_imputed['year_month']=gdf_grace_imputed['year_month'].astype(str).str[:7]
gdf_grace_imputed=gdf_grace_imputed.drop(columns=[
   'location_id','mid_month_date','lat','lon','lwe_obs','lwe_lag1','lwe_lag2',
   'lwe_lag3','month'],errors='ignore')
if gdf_wells.crs != gdf_grace_imputed.crs:
    gdf_wells=gdf_wells.to_crs(gdf_grace_imputed.crs)

# Preserve original per-month polygon join, then retain one hit for each input row.
parts=[]
for ym,wells_ym in gdf_wells.reset_index(drop=True).assign(
        _well_row_id=lambda x:np.arange(len(x))).groupby('year_month',sort=True):
    polygons=gdf_grace_imputed[gdf_grace_imputed['year_month'].eq(ym)]
    if polygons.empty:
        parts.append(wells_ym.drop(columns='_well_row_id'))
    else:
        joined=gpd.sjoin(wells_ym,polygons,how='left',predicate='within')
        joined=(joined.sort_values('_well_row_id')
                .drop_duplicates('_well_row_id',keep='first')
                .drop(columns=['index_right','_well_row_id'],errors='ignore'))
        parts.append(joined)
gdf_wells_grace=gpd.GeoDataFrame(pd.concat(parts,ignore_index=True),crs=gdf_wells.crs)
gdf_wells_grace['date']=pd.to_datetime(gdf_wells_grace['date'],errors='coerce')
gdf=gdf_wells_grace.loc[gdf_wells_grace['date'].between('2003-01-01','2023-12-23')].copy()
gdf=gdf.rename(columns={'grace':'tws_raw','lwe_imputed':'tws_processed'})
# Preserve source month from the dated well observation. Without this cleanup,
# mixing months with and without GRACE polygons can produce duplicate named
# year_month columns after the spatial join.
gdf=gdf.drop(columns=['year_month','year_month_left','year_month_right'],errors='ignore')
gdf['year_month']=gdf['date'].dt.strftime('%Y-%m')
gdf=gdf.merge(df_static[STATIC_COLS],on='well_ID',how='left')
gdf['country']=gdf['well_ID'].astype(str).str[:2]
gdf['transboundary_area']=np.select([
    gdf['country'].isin(['UA','PL']), gdf['country'].eq('LT'),gdf['country'].eq('EE'),
    gdf['country'].isin(['MX','US']),
    (gdf['country'].eq('LV')) & (gdf['Lat_WGS84']>57),
    (gdf['country'].eq('LV')) & (gdf['Lat_WGS84']<57)],
    ['PL-UA transboundary','LT-LV transboundary','LV-EE transboundary',
     'US-MX transboundary','LV-EE transboundary','LT-LV transboundary'],default=None)
for name in ['gw_level_abs','gw_level_from_ground','aquifer_thickness']:
    if name in gdf:
        gdf[name]=pd.to_numeric(gdf[name].astype(str).str.replace(',','.',regex=False),errors='coerce')
gdf['year_month']=pd.to_datetime(gdf['year_month'])
gdf=gdf.sort_values(['country','well_ID','year_month']).reset_index(drop=True)

record_keys=gdf.assign(month=gdf['date'].dt.to_period('M'))
source_qc=pd.DataFrame([{
    'rows':len(gdf),'wells':gdf.well_ID.nunique(),
    'duplicate_well_month_rows':int(record_keys.duplicated(['well_ID','month']).sum()),
    'missing_tws':int(gdf['tws_processed'].isna().sum()),
    'missing_static_lithology':int(gdf['aquifer_lithology'].isna().sum()),
    'missing_transboundary_area':int(gdf['transboundary_area'].isna().sum()),
    'static_duplicate_well_ids':int(df_static.duplicated('well_ID').sum())}])
source_qc.to_csv(TABLE_DIR/'01_source_quality_audit.csv',index=False)
display(source_qc)
if source_qc.loc[0,'duplicate_well_month_rows']:
    warnings.warn('Duplicate well-month records: inspect before interpreting row-based lags.')

## 2. Exploratory data analysis and missingness
The statistics/map notebook excludes wells with **less than 5% of represented monthly rows** containing measured GWL; the modelling notebook instead uses the unfiltered harmonized file and independently discards series that cannot be fitted. These pathways are retained separately. The table below is regenerated from the harmonized input, not a previously exported Parquet file.

In [ ]:
availability=(gdf.assign(month=gdf['date'].dt.to_period('M'))
  .groupby('well_ID').apply(lambda d:d.groupby('month')['gw_level_abs']
                             .apply(lambda s:s.notna().any()).mean())
  .rename('avail_ratio').reset_index())
gdf_stats=gdf[gdf['well_ID'].isin(availability.loc[
    availability['avail_ratio']>=0.05,'well_ID'])].copy()
VARS_MAP={'temp_era5':'Temperature','precip_era5':'Precipitation',
          'aet':'Actual evapotranspiration','pet':'Potential evapotranspiration',
          'tws_processed':'Terrestrial water storage'}
summary_stats=(gdf_stats.groupby('transboundary_area')[list(VARS_MAP)]
    .agg(['count','mean','median','std','min','max'])
    .stack(level=0,future_stack=True).reset_index()
    .rename(columns={'level_1':'Variable'}))
summary_stats['Variable']=summary_stats['Variable'].map(VARS_MAP)
summary_stats.to_csv(TABLE_DIR/'02_area_hydroclimate_statistics.csv',index=False)

# Missing months measured against each well's first/last available row.
vars_ts=[x for x in ['temp_eobs','precip_eobs','temp_era5','precip_era5',
          'aet','pet','tws_processed','gw_level_abs'] if x in gdf]
def gap_summary(d):
    months=d['date'].dt.to_period('M'); expected=len(pd.period_range(months.min(),months.max()))
    row={'expected_months':expected,'observed_months_any':months.nunique()}
    for col in vars_ts:
        count=months[d[col].notna()].nunique()
        row[f'{col}_observed_months']=count
        row[f'{col}_missing_months']=expected-count
        row[f'{col}_missing_fraction']=(expected-count)/expected if expected else np.nan
    return pd.Series(row)
missing_detailed=gdf.groupby(['well_ID','country']).apply(gap_summary).reset_index()
missing_detailed.to_csv(TABLE_DIR/'03_missing_months_by_well.csv',index=False)
display(summary_stats.head(12))

## 3. Engineer hydroclimatic predictors
Exact source-notebook list: contemporaneous T/P/AET/PET/TWS, monthly lags **1/2/3/6/9/12**, rolling means **3/6/9** for all five, additional precipitation rolling sums **3/6/9**, and monthly sine/cosine. Monthly records are sorted by well and time. **Important:** the original lag/rolling implementation uses preceding **rows**, not a reindexed complete monthly grid. Retained deliberately for numerical comparability; check the missing-months diagnostic before interpreting their physical time lag.

In [ ]:
gdf_ml=gdf.copy().sort_values(['well_ID','year_month']).reset_index(drop=True)
VARS_LAG=['temp_era5','precip_era5','aet','pet','tws_processed']
LAGS=(1,2,3,6,9,12); ROLLS=(3,6,9)
for col in VARS_LAG:
    group=gdf_ml.groupby('well_ID')[col]
    for n in LAGS: gdf_ml[f'{col}_lag{n}']=group.shift(n)
    for n in ROLLS:
        gdf_ml[f'{col}_roll{n}_mean']=(group.rolling(n,min_periods=n)
                                        .mean().reset_index(level=0,drop=True))
for n in ROLLS:
    gdf_ml[f'p_roll{n}_sum']=(gdf_ml.groupby('well_ID')['precip_era5']
        .rolling(n,min_periods=n).sum().reset_index(level=0,drop=True))
gdf_ml['month_sin']=np.sin(2*np.pi*gdf_ml['year_month'].dt.month/12)
gdf_ml['month_cos']=np.cos(2*np.pi*gdf_ml['year_month'].dt.month/12)
gdf_ml['gw_level_rel']=gdf_ml['gw_level_abs']-gdf_ml.groupby('well_ID')['gw_level_abs'].transform('mean')
# Explicit list reproduces the original modelling cell, but avoids 50 hard-coded tokens.
CLIMATE_FEATURES=(['aet','pet','precip_era5','temp_era5']
 + [f'{v}_lag{n}' for v in ('temp_era5','precip_era5','aet','pet') for n in LAGS]
 + [f'{v}_roll{n}_mean' for v in ('temp_era5','precip_era5','aet','pet') for n in ROLLS]
 + [f'p_roll{n}_sum' for n in ROLLS]+['month_sin','month_cos'])
TWS_FEATURES=['tws_processed']+[f'tws_processed_lag{n}' for n in LAGS]+[
 f'tws_processed_roll{n}_mean' for n in ROLLS]
FEATURES_STEP1=CLIMATE_FEATURES+TWS_FEATURES
FEATURES_STEP2=FEATURES_STEP1+['gw_level_rel_ref']
assert len(FEATURES_STEP1)==len(set(FEATURES_STEP1))
print(f'{len(FEATURES_STEP1)} predictors in Step 1, {len(FEATURES_STEP2)} in Step 2')

## 4. Supervised ML, Step 1 — preserve original implementation
The original modelling function below is retained **functionally unchanged**. Each well is trimmed to the first/last nonmissing target, split into five chronological segments, with CV on segments 1–3 (three `TimeSeriesSplit` folds), final fitting on segments 1–4, and testing on segment 5. Four candidate models: dummy mean, Ridge α=1, Random Forest (400 trees, `max_features='sqrt'`), and Gradient Boosting (500 estimators, η=0.05, depth 3, subsample 0.9). Winner: highest mean CV R²; SHAP for non-dummy winners.

**Known validation caveat retained:** the source code imputes predictors once on the entire CV development block before internal folds. This is not fold-isolated preprocessing; it can leak predictor-distribution information between CV folds. Results from this notebook remain aligned with the supplied notebooks, rather than silently changing to a stricter pipeline.

In [ ]:
# ============================================================
# Reusable: Monthly ML per well with 5 chronological segments + CV winner
# - Works on any df you pass (gdf_ml-like)
# - Pass feature list + target name
# - Compares: Dummy + Ridge + RandomForest + GradientBoosting
# - 5 segments by record count:
#     seg1–3: CV via TimeSeriesSplit
#     seg5: test (train on seg1–4)
# - Winner per well = highest mean CV R2
# - Collect SHAP for winning model per well
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer

import shap


def run_monthly_ml_per_well(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    *,
    id_col: str = "well_ID",
    date_col: str = "date",
    area_col: str | None = "transboundary_area",
    random_state: int = 42,
    max_shap_samples_per_well: int = 1000,
    min_samples_required: int = 25,
    cv_n_splits: int = 3,
    normalize_monthly: bool = False,   # set True if dates are not aligned to month starts
):
    """
    Returns a dict with:
      - summary: per-well winner + CV/Test metrics
      - results_df: wide metrics table
      - best_model_per_well: {well_id: (model_name, feature_set_name)}
      - cv_scores_all/test_scores_all: raw per-model lists
      - feature_importances_all: tree feature_importances_
      - shap_values_all/shap_importances_all/shap_metadata_all: winner-only
      - skipped_wells: list
    """

    # -----------------------------
    # internal storage
    # -----------------------------
    imputed_all_models = {}
    cv_scores_all = {}
    test_scores_all = {}
    skipped_wells = []
    feature_importances_all = {}
    best_model_per_well = {}
    best_model_params = {}
    cv_masked_intervals = {}

    shap_values_all = {}
    shap_importances_all = {}
    shap_metadata_all = {}

    # -----------------------------
    # helpers
    # -----------------------------
    def rmse(y_true, y_pred) -> float:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

    def split_into_5_segments(df_well: pd.DataFrame) -> list[pd.DataFrame]:
        df_well = df_well.sort_values(date_col).reset_index(drop=True)
        n = len(df_well)
        if n == 0:
            return []
        bounds = np.linspace(0, n, 6).round().astype(int)
        segs = []
        for i in range(5):
            a, b = bounds[i], bounds[i + 1]
            segs.append(df_well.iloc[a:b].copy().reset_index(drop=True))
        return segs

    def get_models(rs: int):
        return {
            "dummy_mean": DummyRegressor(strategy="mean"),
            # Ridge has random_state in sklearn signature but it isn't used; harmless.
            "linear_ridge": Ridge(alpha=1.0, random_state=rs),
            "random_forest": RandomForestRegressor(
                n_estimators=400, random_state=rs, n_jobs=-1, max_features="sqrt"
            ),
            "gradient_boosting": GradientBoostingRegressor(
                random_state=rs, n_estimators=500, learning_rate=0.05, max_depth=3, subsample=0.9
            ),
        }

    def prep_X_y(df_part: pd.DataFrame, feature_cols: list[str]):
        df_part = df_part.copy()
        df_part = df_part[df_part[target].notna()].copy()
        if df_part.empty:
            return None, None, None

        X = df_part[feature_cols]
        y = df_part[target].values

        imputer = SimpleImputer(strategy="median")
        X_imp = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

        return X_imp, y, imputer

    def time_series_cv_scores(X: pd.DataFrame, y: np.ndarray, model_name: str, model, n_splits=3):
        if len(y) < 20:
            return None

        max_splits = min(n_splits, max(2, len(y) // 10))
        if max_splits < 2:
            return None

        tscv = TimeSeriesSplit(n_splits=max_splits)

        r2_list, rmse_list = [], []
        for tr_idx, va_idx in tscv.split(X):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            if model_name == "linear_ridge":
                scaler = StandardScaler().fit(X_tr)
                model.fit(scaler.transform(X_tr), y_tr)
                preds = model.predict(scaler.transform(X_va))
            else:
                model.fit(X_tr, y_tr)
                preds = model.predict(X_va)

            r2_list.append(r2_score(y_va, preds))
            rmse_list.append(rmse(y_va, preds))

        return (
            float(np.mean(r2_list)),
            float(np.std(r2_list, ddof=1)) if len(r2_list) > 1 else 0.0,
            float(np.mean(rmse_list)),
            float(np.std(rmse_list, ddof=1)) if len(rmse_list) > 1 else 0.0,
            int(len(r2_list)),
        )

    def train_test_scores(train_df, test_df, feature_cols, model_name, model):
        X_tr, y_tr, imputer = prep_X_y(train_df, feature_cols)
        if X_tr is None:
            return None

        test_df = test_df.copy()
        test_df = test_df[test_df[target].notna()].copy()
        if test_df.empty:
            return None

        X_te_raw = test_df[feature_cols]
        X_te = pd.DataFrame(imputer.transform(X_te_raw), columns=feature_cols)
        y_te = test_df[target].values

        if model_name == "linear_ridge":
            scaler = StandardScaler().fit(X_tr)
            model.fit(scaler.transform(X_tr), y_tr)
            preds = model.predict(scaler.transform(X_te))
            fitted = (imputer, scaler, model)
        else:
            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)
            fitted = (imputer, None, model)

        return {
            "r2_test": float(r2_score(y_te, preds)),
            "rmse_test": float(rmse(y_te, preds)),
            "n_train": int(len(y_tr)),
            "n_test": int(len(y_te)),
            "fitted": fitted,
            "X_train_imp": X_tr,
            "y_train": y_tr,
        }

    def collect_shap(
        well_id, key, model_name, fitted_bundle, X_train_imp, y_train, feature_cols
    ):
        rng = np.random.default_rng(random_state)
        if len(X_train_imp) > max_shap_samples_per_well:
            idx = rng.choice(len(X_train_imp), size=max_shap_samples_per_well, replace=False)
            Xs = X_train_imp.iloc[idx].reset_index(drop=True)
            ys = y_train[idx]
        else:
            Xs = X_train_imp.reset_index(drop=True)
            ys = y_train

        imputer, scaler, model = fitted_bundle

        # No SHAP for dummy
        if model_name == "dummy_mean":
            return

        # Ridge: use masker instead of deprecated feature_perturbation
        if model_name == "linear_ridge":
            X_scaled = scaler.transform(Xs)
            masker = shap.maskers.Independent(X_scaled)
            explainer = shap.LinearExplainer(model, masker)
            shap_vals = explainer.shap_values(X_scaled)
            base_val = explainer.expected_value

        # Tree models
        else:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(Xs)
            base_val = explainer.expected_value

        shap_vals = np.array(shap_vals)
        mean_abs = np.abs(shap_vals).mean(axis=0)

        shap_values_all.setdefault(key, []).append({
            "well_ID": well_id,
            "feature_names": feature_cols,
            "shap_values": shap_vals,
            "base_value": base_val,
            "X_features": Xs,
            "y": ys,
        })
        shap_importances_all.setdefault(key, []).append(dict(zip(feature_cols, mean_abs)))
        shap_metadata_all.setdefault(key, []).append({
            "well_ID": well_id,
            "n_samples": int(len(Xs)),
            "model_name": model_name,
        })

    # -----------------------------
    # setup
    # -----------------------------
    df_work = df.copy()
    df_work[date_col] = pd.to_datetime(df_work[date_col])
    if normalize_monthly:
        df_work[date_col] = df_work[date_col].dt.to_period("M").dt.to_timestamp("MS")

    # only keep features present in df (pre-filter)
    base_features = [f for f in features if f in df_work.columns]
    if len(base_features) < 2:
        raise ValueError("Too few valid feature columns found in df.")

    feature_sets = {"main": base_features}
    models = get_models(random_state)

    np.random.seed(random_state)

    # -----------------------------
    # main loop per well
    # -----------------------------
    for well_id, group in df_work.groupby(id_col):
        group = group.copy()
        group = group.sort_values(date_col).reset_index(drop=True)

        # skip wells with no targets
        if not group[target].notna().any():
            skipped_wells.append(well_id)
            continue

        # trim to first/last non-null target
        first_dt = group.loc[group[target].notna(), date_col].min()
        last_dt  = group.loc[group[target].notna(), date_col].max()
        group = group[(group[date_col] >= first_dt) & (group[date_col] <= last_dt)].copy()

        data_non_null = group[group[target].notna()].sort_values(date_col)
        if len(data_non_null) < min_samples_required:
            skipped_wells.append(well_id)
            continue

        best_r2 = -np.inf
        best_key = None
        best_params = None

        segs = split_into_5_segments(group)
        if len(segs) < 5 or sum(len(s) for s in segs) < min_samples_required:
            skipped_wells.append(well_id)
            continue

        df_cv = pd.concat(segs[:3], axis=0).reset_index(drop=True)
        df_train_final = pd.concat(segs[:4], axis=0).reset_index(drop=True)
        df_test = segs[4].reset_index(drop=True)

        for feature_set_name, selected_features in feature_sets.items():
            feature_candidates = [f for f in selected_features if f in group.columns]
            if len(feature_candidates) < 2:
                continue

            X_cv, y_cv, _ = prep_X_y(df_cv, feature_candidates)
            if X_cv is None or len(y_cv) < 20:
                continue

            for model_name, model_proto in models.items():
                key = (model_name, feature_set_name)

                cv_scores_all.setdefault(key, [])
                test_scores_all.setdefault(key, [])
                imputed_all_models.setdefault(key, [])
                feature_importances_all.setdefault(key, [])

                # CV on seg1–3
                cv_res = time_series_cv_scores(X_cv, y_cv, model_name, model_proto, n_splits=cv_n_splits)
                if cv_res is None:
                    continue

                r2_mean, r2_std, rmse_mean, rmse_std, folds = cv_res
                cv_scores_all[key].append((well_id, r2_mean, r2_std, rmse_mean, rmse_std, folds))

                if r2_mean > best_r2:
                    best_r2 = r2_mean
                    best_key = key
                    best_params = None

                # Test on seg5 (train seg1–4)
                test_res = train_test_scores(df_train_final, df_test, feature_candidates, model_name, model_proto)
                if test_res is not None:
                    test_scores_all[key].append(
                        (well_id, test_res["r2_test"], test_res["rmse_test"], test_res["n_train"], test_res["n_test"])
                    )

        if best_key is None:
            skipped_wells.append(well_id)
            continue

        best_model_per_well[well_id] = best_key
        best_model_params[well_id] = best_params

        # train winner on seg1–4 and collect SHAP
        model_name, feature_set_name = best_key
        feature_candidates = [f for f in feature_sets[feature_set_name] if f in group.columns]

        winner_model = models[model_name]
        final_fit = train_test_scores(df_train_final, df_test, feature_candidates, model_name, winner_model)
        if final_fit is None:
            skipped_wells.append(well_id)
            continue

        fitted_bundle = final_fit["fitted"]
        X_train_imp = final_fit["X_train_imp"]
        y_train = final_fit["y_train"]

        if model_name in ("random_forest", "gradient_boosting"):
            _, _, est = fitted_bundle
            if hasattr(est, "feature_importances_"):
                feature_importances_all[best_key].append(dict(zip(feature_candidates, est.feature_importances_)))

        collect_shap(
            well_id=well_id,
            key=best_key,
            model_name=model_name,
            fitted_bundle=fitted_bundle,
            X_train_imp=X_train_imp,
            y_train=y_train,
            feature_cols=feature_candidates
        )

    # -----------------------------
    # summary table
    # -----------------------------
    rows = []
    for well_id, best_key in best_model_per_well.items():
        model_name, feature_set_name = best_key

        metrics_cv = next((t for t in cv_scores_all.get(best_key, []) if t[0] == well_id), None)
        if metrics_cv is None:
            _, r2_mean, r2_std, rmse_mean, rmse_std, folds = (well_id, np.nan, np.nan, np.nan, np.nan, np.nan)
        else:
            _, r2_mean, r2_std, rmse_mean, rmse_std, folds = metrics_cv

        metrics_te = next((t for t in test_scores_all.get(best_key, []) if t[0] == well_id), None)
        if metrics_te is None:
            _, r2_test, rmse_test, n_tr, n_te = (well_id, np.nan, np.nan, np.nan, np.nan)
        else:
            _, r2_test, rmse_test, n_tr, n_te = metrics_te

        row = {
            id_col: well_id,
            "best_model": model_name,
            "feature_set_name": feature_set_name,
            "CV_R2 mean": r2_mean,
            "CV_R2 std": r2_std,
            "CV_RMSE mean": rmse_mean,
            "CV_RMSE std": rmse_std,
            "CV_folds": folds,
            "TEST_R2": r2_test,
            "TEST_RMSE": rmse_test,
            "n_train": n_tr,
            "n_test": n_te,
            "best_params": best_model_params.get(well_id),
        }

        if area_col and area_col in df_work.columns:
            row[area_col] = df_work.loc[df_work[id_col] == well_id, area_col].iloc[0]

        rows.append(row)

    summary = pd.DataFrame(rows).sort_values("CV_R2 mean", ascending=False)

    avg = {
        id_col: "AVERAGE",
        "best_model": "",
        "feature_set_name": "",
        "CV_R2 mean": summary["CV_R2 mean"].mean(),
        "CV_R2 std": summary["CV_R2 std"].mean(),
        "CV_RMSE mean": summary["CV_RMSE mean"].mean(),
        "CV_RMSE std": summary["CV_RMSE std"].mean(),
        "CV_folds": summary["CV_folds"].mean(),
        "TEST_R2": summary["TEST_R2"].mean(),
        "TEST_RMSE": summary["TEST_RMSE"].mean(),
        "n_train": summary["n_train"].mean(),
        "n_test": summary["n_test"].mean(),
        "best_params": "",
    }
    if area_col and area_col in df_work.columns:
        avg[area_col] = ""
    summary = pd.concat([summary, pd.DataFrame([avg])], ignore_index=True)

    # -----------------------------
    # wide results_df
    # -----------------------------
    def build_results_df_wide():
        model_names = list(models.keys())
        out_rows = []
        wells = sorted(df_work[id_col].unique())

        cv_lookup = {k: {t[0]: t[1:] for t in v} for k, v in cv_scores_all.items()}
        te_lookup = {k: {t[0]: t[1:] for t in v} for k, v in test_scores_all.items()}

        for w in wells:
            base = {
                id_col: w,
                "status": "ok" if w in best_model_per_well else "skipped",
                "best_key": best_model_per_well.get(w, None),
            }
            if area_col and area_col in df_work.columns:
                base[area_col] = df_work.loc[df_work[id_col] == w, area_col].iloc[0]

            for fs_name in feature_sets.keys():
                for m in model_names:
                    key = (m, fs_name)
                    cvm = cv_lookup.get(key, {}).get(w, None)
                    tem = te_lookup.get(key, {}).get(w, None)

                    if cvm is not None:
                        base[f"{m}__{fs_name}__r2_cv_mean"] = cvm[0]
                        base[f"{m}__{fs_name}__r2_cv_std"] = cvm[1]
                        base[f"{m}__{fs_name}__rmse_cv_mean"] = cvm[2]
                        base[f"{m}__{fs_name}__rmse_cv_std"] = cvm[3]
                        base[f"{m}__{fs_name}__cv_folds"] = cvm[4]

                    if tem is not None:
                        base[f"{m}__{fs_name}__r2_test"] = tem[0]
                        base[f"{m}__{fs_name}__rmse_test"] = tem[1]
                        base[f"{m}__{fs_name}__n_train"] = tem[2]
                        base[f"{m}__{fs_name}__n_test"] = tem[3]

            bk = base["best_key"]
            if bk is not None:
                bm, bfs = bk
                base["best_model"] = bm
                base["best_feature_set"] = bfs
                base["best__r2_cv_mean"] = base.get(f"{bm}__{bfs}__r2_cv_mean", np.nan)
                base["best__rmse_cv_mean"] = base.get(f"{bm}__{bfs}__rmse_cv_mean", np.nan)
                base["best__r2_test"] = base.get(f"{bm}__{bfs}__r2_test", np.nan)
                base["best__rmse_test"] = base.get(f"{bm}__{bfs}__rmse_test", np.nan)

            out_rows.append(base)

        return pd.DataFrame(out_rows).sort_values(["status", id_col]).reset_index(drop=True)

    results_df = build_results_df_wide()

    return {
        "summary": summary,
        "results_df": results_df,
        "best_model_per_well": best_model_per_well,
        "best_model_params": best_model_params,
        "cv_scores_all": cv_scores_all,
        "test_scores_all": test_scores_all,
        "feature_importances_all": feature_importances_all,
        "shap_values_all": shap_values_all,
        "shap_importances_all": shap_importances_all,
        "shap_metadata_all": shap_metadata_all,
        "skipped_wells": skipped_wells,
        "cv_masked_intervals": cv_masked_intervals,
        "imputed_all_models": imputed_all_models,
    }

In [ ]:
out=run_monthly_ml_per_well(gdf_ml,FEATURES_STEP1,'gw_level_abs',
     id_col='well_ID',date_col='date',area_col='transboundary_area',
     random_state=SEED,min_samples_required=25,cv_n_splits=3,normalize_monthly=False)
summary=step1_summary=out['summary']
step1_results_df=out['results_df']
print('Step-1 fitted wells:',(summary['well_ID']!='AVERAGE').sum(),
      '; skipped:',len(out['skipped_wells']))

In [ ]:
import numpy as np
import pandas as pd

def build_baseline_vs_ml_tables(out, df, id_col="well_ID", area_col="transboundary_area"):
    results_df = out["results_df"].copy()
    best_model_per_well = out["best_model_per_well"]

    rows = []
    for _, row in results_df.iterrows():
        well_id = row[id_col]

        # baseline CV R2
        baseline_cv = row.get("dummy_mean__main__r2_cv_mean", np.nan)

        # best model chosen by the pipeline
        best_key = row.get("best_key", None)
        if best_key is None:
            continue

        best_model, best_feature_set = best_key
        best_cv = row.get(f"{best_model}__{best_feature_set}__r2_cv_mean", np.nan)

        # best non-baseline CV R2
        candidate_cols = [c for c in results_df.columns if c.endswith("__r2_cv_mean") and not c.startswith("dummy_mean__")]
        nonbaseline_vals = row[candidate_cols].dropna()
        best_nonbaseline_cv = nonbaseline_vals.max() if len(nonbaseline_vals) else np.nan

        best_nonbaseline_model = None
        if len(nonbaseline_vals):
            best_nonbaseline_model = nonbaseline_vals.idxmax().replace("__r2_cv_mean", "")

        area_val = None
        if area_col in df.columns:
            area_series = df.loc[df[id_col] == well_id, area_col]
            area_val = area_series.iloc[0] if len(area_series) else None

        rows.append({
            id_col: well_id,
            area_col: area_val,
            "baseline_cv_r2": baseline_cv,
            "winner_cv_r2": best_cv,
            "winner_model": best_model,
            "best_nonbaseline_cv_r2": best_nonbaseline_cv,
            "best_nonbaseline_model": best_nonbaseline_model,
            "delta_cv_r2": best_nonbaseline_cv - baseline_cv if pd.notna(best_nonbaseline_cv) and pd.notna(baseline_cv) else np.nan,
            "ml_beats_baseline": (
                (best_nonbaseline_cv > baseline_cv)
                if pd.notna(best_nonbaseline_cv) and pd.notna(baseline_cv)
                else np.nan
            )
        })

    per_well = pd.DataFrame(rows)

    if area_col in per_well.columns:
        per_area = (
            per_well.groupby(area_col, dropna=False)
            .agg(
                n_wells=(id_col, "count"),
                mean_baseline_cv_r2=("baseline_cv_r2", "mean"),
                median_baseline_cv_r2=("baseline_cv_r2", "median"),
                mean_best_nonbaseline_cv_r2=("best_nonbaseline_cv_r2", "mean"),
                median_best_nonbaseline_cv_r2=("best_nonbaseline_cv_r2", "median"),
                mean_delta_cv_r2=("delta_cv_r2", "mean"),
                median_delta_cv_r2=("delta_cv_r2", "median"),
                wells_ml_beats_baseline=("ml_beats_baseline", lambda s: np.nansum(s.astype(float))),
                share_ml_beats_baseline=("ml_beats_baseline", lambda s: np.nanmean(s.astype(float))),
            )
            .reset_index()
            .sort_values("mean_delta_cv_r2", ascending=False)
        )
    else:
        per_area = pd.DataFrame()

    overall = pd.DataFrame([{
        "n_wells": len(per_well),
        "mean_baseline_cv_r2": per_well["baseline_cv_r2"].mean(),
        "mean_best_nonbaseline_cv_r2": per_well["best_nonbaseline_cv_r2"].mean(),
        "mean_delta_cv_r2": per_well["delta_cv_r2"].mean(),
        "median_delta_cv_r2": per_well["delta_cv_r2"].median(),
        "wells_ml_beats_baseline": np.nansum(per_well["ml_beats_baseline"].astype(float)),
        "share_ml_beats_baseline": np.nanmean(per_well["ml_beats_baseline"].astype(float)),
    }])

    return per_well, per_area, overall


per_well_baseline_vs_ml, per_area_baseline_vs_ml, overall_baseline_vs_ml = build_baseline_vs_ml_tables(
    out=out,
    df=gdf_ml,
    id_col="well_ID",
    area_col="transboundary_area"
)

print("Overall baseline vs ML comparison")
display(overall_baseline_vs_ml)

print("Per transboundary area")
display(per_area_baseline_vs_ml)

print("Per well")
display(per_well_baseline_vs_ml.sort_values("delta_cv_r2", ascending=False))

## 5. Reference wells and supervised ML, Step 2
Select the highest mean Step-1 CV R² in each area among wells with **at least 96 training+test observations**, breaking ties by smaller CV R² standard deviation. Fit that well's winning model on all observed months and reconstruct its complete series. Subtract the reconstructed series mean to obtain a relative reference signal, then merge on the **same exact dates and area**.

**Known holdout caveat retained:** because the original reference signal is fitted using **all** reference-well observations, including later dates, its use in Step-2 testing is not a fully out-of-sample nested transfer test. The reference well itself is excluded from reported inter-well improvement counts. The notebook documents the resulting dependence; it does not claim leak-free Step-2 holdout evaluation.

In [ ]:
summary_best = (
    summary[summary["well_ID"] != "AVERAGE"]
    .dropna(subset=["CV_R2 mean", "CV_R2 std", "transboundary_area", "n_train", "n_test"])
    .copy()
)

# total number of target records used by the winning model
summary_best["n_total_used"] = summary_best["n_train"] + summary_best["n_test"]

# keep only wells with at least 96 entries
summary_best = summary_best[summary_best["n_total_used"] >= 96].copy()

best_well_by_area = (
    summary_best
    .sort_values(
        by=["transboundary_area", "CV_R2 mean", "CV_R2 std"],
        ascending=[True, False, True]
    )
    .groupby("transboundary_area", as_index=False)
    .first()[[
        "transboundary_area",
        "well_ID",
        "best_model",
        "feature_set_name",
        "CV_R2 mean",
        "CV_R2 std",
        "n_train",
        "n_test",
        "n_total_used",
    ]]
)

best_well_by_area

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def fit_and_impute_full_series_for_well(
    df_well: pd.DataFrame,
    feature_cols: list[str],
    target: str,
    model_name: str,
    model_proto,
    date_col: str = "date",
    random_state: int = 42,
):
    """
    Fits the given model on all rows where target is notna,
    then predicts/imputes target for ALL rows in df_well (full timeline).
    Returns df_well with a new column: f"{target}_imputed".
    """

    df_well = df_well.sort_values(date_col).copy()

    # --- train on available targets
    train_mask = df_well[target].notna()
    if not train_mask.any():
        # nothing to train; return all NaN imputed
        df_well[f"{target}_imputed"] = np.nan
        return df_well

    X_train_raw = df_well.loc[train_mask, feature_cols]
    y_train = df_well.loc[train_mask, target].values

    # impute features
    imputer = SimpleImputer(strategy="median")
    X_train = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=feature_cols)

    # --- prepare ALL rows features for prediction
    X_all_raw = df_well[feature_cols]
    X_all = pd.DataFrame(imputer.transform(X_all_raw), columns=feature_cols)

    # --- fit + predict
    if model_name == "dummy_mean":
        y_pred_all = np.full(len(df_well), float(np.mean(y_train)))

    elif model_name == "linear_ridge":
        scaler = StandardScaler().fit(X_train)
        model = model_proto
        model.fit(scaler.transform(X_train), y_train)
        y_pred_all = model.predict(scaler.transform(X_all))

    else:
        model = model_proto
        model.fit(X_train, y_train)
        y_pred_all = model.predict(X_all)

    df_well[f"{target}_imputed"] = y_pred_all
    return df_well

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def get_models(rs: int):
    return {
        "dummy_mean": DummyRegressor(strategy="mean"),
        "linear_ridge": Ridge(alpha=1.0, random_state=rs),
        "random_forest": RandomForestRegressor(
            n_estimators=400, random_state=rs, n_jobs=-1, max_features="sqrt"
        ),
        "gradient_boosting": GradientBoostingRegressor(
            random_state=rs, n_estimators=500, learning_rate=0.05, max_depth=3, subsample=0.9
        ),
    }

def build_area_reference_relative_series(
    df: pd.DataFrame,
    best_well_by_area: pd.DataFrame,
    features: list[str],
    target: str = "gw_level_abs",
    id_col: str = "well_ID",
    date_col: str = "date",
    area_col: str = "transboundary_area",
    random_state: int = 42,
):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # keep only valid feature columns present
    feature_cols = [f for f in features if f in df.columns]
    if len(feature_cols) < 2:
        raise ValueError("Too few valid feature columns found in df.")

    models = get_models(random_state)

    area_ref_rows = []
    for _, row in best_well_by_area.iterrows():
        area = row[area_col]
        ref_well = row[id_col]
        model_name = row["best_model"]          # from your table
        # feature_set_name exists but you only have "main" right now; ignoring is fine.

        df_ref = df[(df[area_col] == area) & (df[id_col] == ref_well)].copy()
        if df_ref.empty:
            continue

        df_ref = fit_and_impute_full_series_for_well(
            df_well=df_ref,
            feature_cols=feature_cols,
            target=target,
            model_name=model_name,
            model_proto=models[model_name],
            date_col=date_col,
            random_state=random_state,
        )

        # relative = abs_imputed - mean(abs_imputed)
        mean_gwl = df_ref[f"{target}_imputed"].mean()
        df_ref["gw_level_rel_ref"] = df_ref[f"{target}_imputed"] - mean_gwl

        # keep only join keys + reference columns
        area_ref_rows.append(
            df_ref[[date_col, area_col, "gw_level_rel_ref"]]
            .assign(ref_well_ID=ref_well, ref_model=model_name)
        )

    if not area_ref_rows:
        return pd.DataFrame(columns=[date_col, area_col, "gw_level_rel_ref", "ref_well_ID", "ref_model"])

    area_ref = pd.concat(area_ref_rows, ignore_index=True)

    # if duplicates per (date, area) occur (shouldn't, but just in case)
    area_ref = (
        area_ref.sort_values([area_col, date_col])
                .drop_duplicates([area_col, date_col], keep="first")
                .reset_index(drop=True)
    )
    return area_ref

In [ ]:
area_ref = build_area_reference_relative_series(
    df=gdf_ml,
    best_well_by_area=best_well_by_area,
    features=FEATURES_STEP1,
    target="gw_level_abs",
    id_col="well_ID",
    date_col="date",
    area_col="transboundary_area",
    random_state=42,
)

gdf_ml_with_ref = (
    gdf_ml.copy()
          .assign(date=pd.to_datetime(gdf_ml["date"]))
          .merge(area_ref, on=["date", "transboundary_area"], how="left")
)

# Now every row has:
# - gw_level_rel_ref : the imputed-relative level of the best well for that row's area/date
# - ref_well_ID      : which well was used as reference for that area
# - ref_model        : which model was used

In [ ]:
out2=run_monthly_ml_per_well(gdf_ml_with_ref,FEATURES_STEP2,'gw_level_abs',
    id_col='well_ID',date_col='date',area_col='transboundary_area',
    random_state=SEED,min_samples_required=25,cv_n_splits=3,normalize_monthly=False)
summary2=step2_summary=out2['summary']
results_df=out2['results_df']
print('Step-2 fitted wells:',(summary2['well_ID']!='AVERAGE').sum(),
      '; skipped:',len(out2['skipped_wells']))

## 6. Model comparison, test predictions and downstream ML tables
The original prediction-export helpers are reused to reconstruct the winning model's **segment-5** observations/predictions in each step. The reference-well identity comes from the explicit **≥96 months** table, *not* a subsequent unrestricted re-ranking. We also calculate every table in memory for manuscript figures.

In [ ]:
# ============================================================
# Winner test-set predictions for step 1 and step 2
# Reconstruct observed vs predicted values on seg5 for the
# winning model of each well using the existing summary tables.
# Also builds a step1-step2 comparison table and saves outputs.
# ============================================================

from pathlib import Path

def _split_into_5_segments_for_export(df_well, date_col="date"):
    df_well = df_well.sort_values(date_col).reset_index(drop=True)
    n = len(df_well)
    if n == 0:
        return []
    bounds = np.linspace(0, n, 6).round().astype(int)
    return [df_well.iloc[bounds[i]:bounds[i+1]].copy().reset_index(drop=True) for i in range(5)]

def _fit_predict_test_for_named_model(df, well_id, feature_cols, model_name,
                                      target="gw_level_abs", id_col="well_ID",
                                      date_col="date", random_state=42):
    dfw = df[df[id_col] == well_id].copy()
    if dfw.empty:
        return None

    dfw[date_col] = pd.to_datetime(dfw[date_col])
    dfw = dfw.sort_values(date_col).reset_index(drop=True)

    # restrict to effective observed period
    mask_obs = dfw[target].notna()
    if not mask_obs.any():
        return None
    first_dt = dfw.loc[mask_obs, date_col].min()
    last_dt = dfw.loc[mask_obs, date_col].max()
    dfw = dfw[(dfw[date_col] >= first_dt) & (dfw[date_col] <= last_dt)].copy()

    segs = _split_into_5_segments_for_export(dfw, date_col=date_col)
    if len(segs) < 5:
        return None

    df_train = pd.concat(segs[:4], axis=0).reset_index(drop=True)
    df_test = segs[4].copy().reset_index(drop=True)
    df_train = df_train[df_train[target].notna()].copy()
    df_test = df_test[df_test[target].notna()].copy()

    if df_train.empty or df_test.empty:
        return None

    feature_cols = [f for f in feature_cols if f in dfw.columns]
    if len(feature_cols) < 2:
        return None

    X_tr_raw = df_train[feature_cols]
    X_te_raw = df_test[feature_cols]
    y_tr = df_train[target].values
    y_te = df_test[target].values

    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(X_tr_raw), columns=feature_cols, index=df_train.index)
    X_te = pd.DataFrame(imputer.transform(X_te_raw), columns=feature_cols, index=df_test.index)

    models = get_models(random_state)
    model = models[model_name]

    if model_name == "linear_ridge":
        scaler = StandardScaler().fit(X_tr)
        model.fit(scaler.transform(X_tr), y_tr)
        preds = model.predict(scaler.transform(X_te))
    else:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)

    out_df = df_test[[id_col, date_col]].copy()
    if "transboundary_area" in df_test.columns:
        out_df["transboundary_area"] = df_test["transboundary_area"].values
    out_df["y_obs"] = y_te
    out_df["y_pred"] = preds
    out_df["model_name"] = model_name
    return out_df

def build_winner_test_predictions(df, summary_df, feature_cols,
                                  target="gw_level_abs", id_col="well_ID",
                                  date_col="date", random_state=42):
    rows = []
    summary_valid = summary_df.copy()
    if id_col in summary_valid.columns:
        summary_valid = summary_valid[summary_valid[id_col] != "AVERAGE"].copy()

    for _, row in summary_valid.iterrows():
        well_id = row[id_col]
        model_name = row.get("best_model")
        if pd.isna(model_name):
            continue
        pred_df = _fit_predict_test_for_named_model(
            df=df,
            well_id=well_id,
            feature_cols=feature_cols,
            model_name=model_name,
            target=target,
            id_col=id_col,
            date_col=date_col,
            random_state=random_state,
        )
        if pred_df is not None:
            rows.append(pred_df)

    if not rows:
        return pd.DataFrame(columns=[id_col, date_col, "transboundary_area", "y_obs", "y_pred", "model_name"])

    return pd.concat(rows, ignore_index=True).sort_values([id_col, date_col]).reset_index(drop=True)

def build_step1_step2_comparison_table_from_summaries(summary1, summary2, ref_wells_df,
                                                      id_col="well_ID", area_col="transboundary_area"):
    s1 = summary1.copy()
    s2 = summary2.copy()
    s1 = s1[s1[id_col] != "AVERAGE"].copy()
    s2 = s2[s2[id_col] != "AVERAGE"].copy()

    keep = [id_col, area_col, "best_model", "CV_R2 mean", "CV_R2 std", "TEST_R2", "TEST_RMSE", "n_train", "n_test"]

    s1 = s1[keep].rename(columns={
        "best_model": "step1_best_model",
        "CV_R2 mean": "step1_CV_R2_mean",
        "CV_R2 std": "step1_CV_R2_std",
        "TEST_R2": "step1_TEST_R2",
        "TEST_RMSE": "step1_TEST_RMSE",
        "n_train": "step1_n_train",
        "n_test": "step1_n_test",
    })
    s2 = s2[keep].rename(columns={
        "best_model": "step2_best_model",
        "CV_R2 mean": "step2_CV_R2_mean",
        "CV_R2 std": "step2_CV_R2_std",
        "TEST_R2": "step2_TEST_R2",
        "TEST_RMSE": "step2_TEST_RMSE",
        "n_train": "step2_n_train",
        "n_test": "step2_n_test",
    })

    cmp = s1.merge(s2, on=[id_col, area_col], how="outer")
    cmp["delta_CV_R2_mean"] = cmp["step2_CV_R2_mean"] - cmp["step1_CV_R2_mean"]
    cmp["delta_TEST_R2"] = cmp["step2_TEST_R2"] - cmp["step1_TEST_R2"]

    ref_set = set()
    ref_col = None
    for candidate in ["reference_well_ID", "ref_well_ID", "well_ID"]:
        if candidate in ref_wells_df.columns:
            ref_col = candidate
            break
    if ref_col is not None:
        ref_set = set(ref_wells_df[ref_col].astype(str))
    cmp["is_reference_well"] = cmp[id_col].astype(str).isin(ref_set)

    return cmp.sort_values([area_col, "step2_CV_R2_mean"], ascending=[True, False]).reset_index(drop=True)

In [ ]:
winner_test_predictions_step1=build_winner_test_predictions(
    gdf_ml,step1_summary,FEATURES_STEP1)
winner_test_predictions_step2=build_winner_test_predictions(
    gdf_ml_with_ref,step2_summary,FEATURES_STEP2)
comparison_df=build_step1_step2_comparison_table_from_summaries(
    step1_summary,step2_summary,best_well_by_area)
reference_well_ids=set(best_well_by_area['well_ID'].astype(str))
obs_pred_df=pd.concat([
    winner_test_predictions_step1.assign(prediction_step='step1'),
    winner_test_predictions_step2.assign(prediction_step='step2')],ignore_index=True)
# Dataset used in original plotted comparison (suffix naming retained)
summary2_renamed=summary2.rename(columns={col:f'{col}_round2' for col in summary2 if col!='well_ID'})
merged=summary.merge(summary2_renamed,on='well_ID',how='inner')
merged=merged[merged['well_ID']!='AVERAGE'].copy()
ref_tbl=best_well_by_area[['transboundary_area','well_ID','CV_R2 mean']].rename(
  columns={'well_ID':'ref_well_ID','CV_R2 mean':'ref_r1_cv_r2'})
merged_nr=merged[~merged['well_ID'].astype(str).isin(reference_well_ids)].copy()
merged_nr['delta_CV_R2']=merged_nr['CV_R2 mean_round2']-merged_nr['CV_R2 mean']
merged_nr['increased']=merged_nr['delta_CV_R2']>0
area_inc=(merged_nr.groupby('transboundary_area',dropna=False).agg(
 n_wells=('well_ID','count'),n_increased=('increased','sum'),
 share_increased=('increased','mean'),mean_delta=('delta_CV_R2','mean')).reset_index())
area_inc['share_increased_pct']=area_inc['share_increased']*100
# For Fig 6 use EXACT original objective selection: among non-reference wells with
# both steps and >24 common test dates, choose largest Step-2 CV R² per area.
plot_base=obs_pred_df.merge(merged[['well_ID','CV_R2 mean','CV_R2 mean_round2']],
                             on='well_ID',how='left')
plot_base=plot_base[~plot_base['well_ID'].astype(str).isin(reference_well_ids)]
well_rank=(plot_base.groupby(['transboundary_area','well_ID'],as_index=False)
 .agg(n_dates=('date','nunique'),n_steps=('prediction_step','nunique'),
      cv_r2_step2=('CV_R2 mean_round2','first')))
well_rank=well_rank[(well_rank.n_steps>=2)&(well_rank.n_dates>24)]
sample_wells=(well_rank.sort_values(['transboundary_area','cv_r2_step2','n_dates','well_ID'],
    ascending=[True,False,False,True]).groupby('transboundary_area',as_index=False).first())
comparison_df.to_csv(TABLE_DIR/'04_step1_step2_well_comparison.csv',index=False)
area_inc.to_csv(TABLE_DIR/'05_area_level_improvement.csv',index=False)
best_well_by_area.to_csv(TABLE_DIR/'06_reference_well_selection.csv',index=False)
sample_wells.to_csv(TABLE_DIR/'07_figure6_sample_selection.csv',index=False)
step1_summary.to_csv(TABLE_DIR/'08_step1_per_well.csv',index=False)
step2_summary.to_csv(TABLE_DIR/'09_step2_per_well.csv',index=False)
per_well_baseline_vs_ml.to_csv(TABLE_DIR/'10_baseline_vs_ml.csv',index=False)
display(area_inc)
display(sample_wells[['transboundary_area','well_ID','n_dates','cv_r2_step2']])

## 7. Static descriptors and clustering to interpret predictability
Uses the original PCA + K-means diagnostic implementation (up to 5 PCA components, silhouette k=2…min(12,n−1), seed 42) separately in each TBA. Within-area `OK` means Step-2 mean CV R² at/above the 75th percentile **and** CV R² standard deviation at/below its median. Cramér's V and standardized contingency residuals characterize categorical associations; they do **not** establish causality.

In [ ]:
static_attrs=['well_ID','aquifer_thickness','aquifer_lithology','aquifer_karst',
              'confinement','ground_altitude','well_depth','pumping']
joined_cols=['well_ID','best_model','feature_set_name','n_train','n_test','best_params',
             'transboundary_area','best_model_round2','feature_set_name_round2',
             'CV_R2 mean_round2','CV_R2 std_round2','CV_RMSE mean_round2',
             'CV_RMSE std_round2','CV_folds_round2','TEST_R2_round2','TEST_RMSE_round2']
df_joined=merged_nr[[x for x in joined_cols if x in merged_nr]].merge(
  gdf_ml[static_attrs].drop_duplicates('well_ID'),on='well_ID',how='left')

In [ ]:
# ============================================================
# PER-TRANSBOUNDARY-AREA UNSUPERVISED WELL DIAGNOSTICS (LEAN)
# Single self-contained code block
#
# What this version does differently:
# - does NOT build/save per-area objects in `all_outputs`
# - instead builds combined result tables directly during the loop
# - each output table gets a transboundary_area column
#
# Combined outputs:
#   1) lean_all_areas_labeled.csv
#   2) lean_all_areas_cluster_summary.csv
#   3) lean_all_areas_cat_assoc_cramersV.csv
#   4) lean_all_areas_okbad_contrib_stdresid.csv
#   5) lean_all_areas_area_summary.csv
#   6) lean_all_areas_silhouette_scores.csv
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from scipy.stats import chi2_contingency


# ----------------------------
# Helpers
# ----------------------------
def cramers_v(ct: pd.DataFrame) -> float:
    if ct.size == 0:
        return np.nan

    chi2, _, _, _ = chi2_contingency(ct.values)
    n = ct.values.sum()
    if n == 0:
        return np.nan

    r, k = ct.shape
    denom = n * min(r - 1, k - 1)
    return float(np.sqrt(chi2 / denom)) if denom > 0 else np.nan


def std_residuals(ct: pd.DataFrame) -> pd.DataFrame:
    obs = ct.values.astype(float)
    n = obs.sum()
    if n == 0:
        return ct * np.nan

    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)
    exp = (row_sums @ col_sums) / n

    with np.errstate(divide="ignore", invalid="ignore"):
        sr = (obs - exp) / np.sqrt(exp)

    return pd.DataFrame(sr, index=ct.index, columns=ct.columns)


def cat_assoc_cramers(res: pd.DataFrame, cat_cols, cluster_col="cluster"):
    rows = []

    for c in cat_cols:
        if c not in res.columns:
            continue

        d = res[[cluster_col, c]].copy()
        d[c] = d[c].fillna("NA").astype(str)

        ct = pd.crosstab(d[cluster_col], d[c], dropna=False)
        rows.append({
            "categorical_feature": c,
            "cramers_v": cramers_v(ct),
            "n_levels": ct.shape[1],
        })

    if not rows:
        return pd.DataFrame(columns=["categorical_feature", "cramers_v", "n_levels"])

    return pd.DataFrame(rows).sort_values("cramers_v", ascending=False).reset_index(drop=True)


def okbad_contrib(res: pd.DataFrame, cat_cols, ok_col="ok_model", top_levels=15, min_count=3):
    rows = []

    for c in cat_cols:
        if c not in res.columns:
            continue

        d = res[[ok_col, c]].copy()
        d[c] = d[c].fillna("NA").astype(str)

        counts = d[c].value_counts(dropna=False)
        keep = counts[counts >= min_count].index
        d.loc[~d[c].isin(keep), c] = "RARE"

        ct = pd.crosstab(d[ok_col], d[c], dropna=False)
        sr = std_residuals(ct)

        long = sr.stack().reset_index()
        long.columns = ["group", "level", "std_resid"]
        long["categorical_feature"] = c
        long["abs_std_resid"] = long["std_resid"].abs()

        rows.append(long.sort_values("abs_std_resid", ascending=False).head(top_levels))

    if not rows:
        return pd.DataFrame(columns=["group", "level", "std_resid", "categorical_feature", "abs_std_resid"])

    return pd.concat(rows, ignore_index=True)


# ----------------------------
# Per-area lean run
# ----------------------------
def run_area_lean(
    area_df: pd.DataFrame,
    area_name: str,
    id_col: str,
    r2_col: str,
    r2std_col: str,
    num_cols,
    cat_cols,
    pca_components=5,
    k_min=2,
    k_max=12,
    ok_r2_quantile=0.75,
    ok_std_quantile=0.50,
    random_state=42,
    okbad_top_levels=15,
):
    df = area_df.copy()
    df[id_col] = df[id_col].astype(str)
    df[r2_col] = pd.to_numeric(df[r2_col], errors="coerce")

    if r2std_col in df.columns:
        df[r2std_col] = pd.to_numeric(df[r2std_col], errors="coerce")

    n = len(df)
    if n < 8:
        return None

    num_cols = [c for c in num_cols if c in df.columns]
    cat_cols = [c for c in cat_cols if c in df.columns]

    if len(num_cols) + len(cat_cols) == 0:
        return None

    k_hi = min(k_max, n - 1)

    # clustering features: static + completeness only
    Xdf = df[num_cols + cat_cols].copy()
    for c in num_cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")

    pre = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                num_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                cat_cols,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    n_comp = int(min(pca_components, max(2, n - 1)))
    Z = Pipeline([
        ("prep", pre),
        ("pca", PCA(n_components=n_comp, random_state=random_state)),
    ]).fit_transform(Xdf)

    # OK / BAD within area
    r2_cut = df[r2_col].quantile(ok_r2_quantile) if df[r2_col].notna().any() else np.nan

    if r2std_col in df.columns and df[r2std_col].notna().any():
        std_cut = df[r2std_col].quantile(ok_std_quantile)
        df["ok_model"] = (df[r2_col] >= r2_cut) & (df[r2std_col] <= std_cut)
    else:
        std_cut = np.nan
        df["ok_model"] = (df[r2_col] >= r2_cut)

    # choose K by silhouette
    sil_scores = {}
    best_k = None

    if k_hi >= k_min:
        for k in range(k_min, k_hi + 1):
            labels = KMeans(n_clusters=k, n_init="auto", random_state=random_state).fit_predict(Z)

            if len(np.unique(labels)) < 2 or len(np.unique(labels)) >= n:
                continue

            sil_scores[k] = float(silhouette_score(Z, labels))

        if sil_scores:
            best_k = max(sil_scores, key=sil_scores.get)

    if best_k is None:
        return None

    df["cluster"] = KMeans(
        n_clusters=best_k,
        n_init="auto",
        random_state=random_state
    ).fit_predict(Z)

    # cluster summary
    agg = {
        "n_wells": (id_col, "count"),
        "r2_mean": (r2_col, "mean"),
        "frac_ok": ("ok_model", "mean"),
        "n_train_mean": ("n_train", "mean") if "n_train" in df.columns else (id_col, "count"),
        "n_test_mean": ("n_test", "mean") if "n_test" in df.columns else (id_col, "count"),
    }

    if r2std_col in df.columns:
        agg["r2std_mean"] = (r2std_col, "mean")
    else:
        agg["r2std_mean"] = (r2_col, lambda x: np.nan)

    cluster_summary = (
        df.groupby("cluster", dropna=False)
          .agg(**agg)
          .sort_values(["r2_mean", "n_wells"], ascending=[False, False])
          .reset_index()
    )

    cat_assoc = cat_assoc_cramers(df, cat_cols=cat_cols, cluster_col="cluster")
    okbad = okbad_contrib(df, cat_cols=cat_cols, ok_col="ok_model", top_levels=okbad_top_levels)

    area_summary = pd.DataFrame([{
        "transboundary_area": area_name,
        "n_wells": n,
        "best_k": best_k,
        "ok_r2_cut": r2_cut,
        "ok_std_cut": std_cut,
        "n_clusters_found": df["cluster"].nunique(dropna=True),
    }])

    sil_df = pd.DataFrame({
        "transboundary_area": area_name,
        "k": list(sil_scores.keys()),
        "silhouette_score": list(sil_scores.values()),
    }).sort_values("k").reset_index(drop=True)

    return {
        "area_summary": area_summary,
        "silhouette_scores": sil_df,
        "labeled": df,
        "cluster_summary": cluster_summary,
        "cat_assoc": cat_assoc,
        "okbad_contrib": okbad,
    }

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import chi2_contingency

NUM_COLS=['aquifer_thickness','ground_altitude','well_depth','n_train','n_test']
CAT_COLS=['aquifer_lithology','aquifer_karst','confinement','pumping']
cluster_runs={}
for area in AREA_ORDER:
    subset=df_joined[df_joined['transboundary_area'].eq(area)].copy()
    if subset.empty: continue
    result=run_area_lean(subset,area,id_col='well_ID',
        r2_col='CV_R2 mean_round2',r2std_col='CV_R2 std_round2',
        num_cols=NUM_COLS,cat_cols=CAT_COLS,pca_components=5,
        k_min=2,k_max=12,ok_r2_quantile=.75,ok_std_quantile=.5,
        random_state=SEED,okbad_top_levels=15)
    if result: cluster_runs[area]=result
    else: warnings.warn(f'Clustering skipped for {area}: insufficient valid wells/features')

def combine_cluster_table(key):
    parts=[]
    for area,item in cluster_runs.items():
        df=item[key].copy(); df['transboundary_area']=area; parts.append(df)
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
lean_all_areas_labeled=combine_cluster_table('labeled')
lean_all_areas_cluster_summary=combine_cluster_table('cluster_summary')
lean_all_areas_cat_assoc_cramersV=combine_cluster_table('cat_assoc')
lean_all_areas_okbad_contrib_stdresid=combine_cluster_table('okbad_contrib')
lean_all_areas_area_summary=combine_cluster_table('area_summary')
lean_all_areas_silhouette_scores=combine_cluster_table('silhouette_scores')
for filename,table in {
 '11_cluster_assignment.csv':lean_all_areas_labeled,
 '12_cluster_summary.csv':lean_all_areas_cluster_summary,
 '13_cluster_categorical_cramersV.csv':lean_all_areas_cat_assoc_cramersV,
 '14_okbad_standardized_residuals.csv':lean_all_areas_okbad_contrib_stdresid,
 '15_cluster_area_summary.csv':lean_all_areas_area_summary,
 '16_silhouette_scores.csv':lean_all_areas_silhouette_scores}.items():
    table.to_csv(TABLE_DIR/filename,index=False)
display(lean_all_areas_area_summary)

## 8. Extract Step-2 SHAP values for publication
Flatten the winner-only SHAP dictionaries directly. `feature_value` is the fitted/imputed predictor value, as in the source's plotting export. This reconstruction does not require old `shap_values_long.parquet` links. Values are subsampled at **up to 1000** points per well in the original model code.

In [ ]:
def shap_to_long(model_result):
    frames=[]
    for key,items in model_result.get('shap_values_all',{}).items():
        for item in items or []:
            names=list(item['feature_names']); values=np.asarray(item['shap_values'])
            if values.ndim==1:values=values[None,:]
            features=pd.DataFrame(item['X_features']).reindex(columns=names)
            n=min(len(features),len(values))
            if n==0:continue
            val_df=pd.DataFrame(values[:n],columns=names).stack().rename('shap_value').reset_index()
            val_df.columns=['sample_idx','feature_name','shap_value']
            feat_df=features.iloc[:n].reset_index(drop=True).stack().rename('feature_value').reset_index()
            feat_df.columns=['sample_idx','feature_name','feature_value']
            frame=val_df.merge(feat_df,on=['sample_idx','feature_name'])
            frame['well_ID']=str(item['well_ID']);frame['model_key']=str(key)
            frames.append(frame)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=[
        'well_ID','model_key','sample_idx','feature_name','shap_value','feature_value'])
shap_long_df=shap_to_long(out2)
shap_long_df.to_parquet(TABLE_DIR/'17_step2_shap_values_long.parquet',index=False)
print('Step-2 SHAP rows:',len(shap_long_df))

---
# Manuscript outputs, **in manuscript order**
All inputs have now been loaded/analysed; no external file reads occur below. Output names are `Figure_01` to `Figure_09` followed by appendix figure/table files. Each graphic is also available as 600-dpi JPEG and, where appropriate, vector PDF.

### Figure 1 — Geographic distribution of four transboundary systems
Reconstructed from the original three-panel world/US–MX/European zoom code. Country boundaries are taken from the **original map GeoJSON input**. Well locations use the ≥5% completeness filter applied by the statistics/map notebook.

In [ ]:
world_4326=world.to_crs('EPSG:4326') if world.crs else world.set_crs('EPSG:4326')
gdf_4326=gdf_stats.to_crs('EPSG:4326')
zoom_usmx={'xmin':-115,'xmax':-95,'ymin':20,'ymax':40}
zoom_europe={'xmin':15,'xmax':35,'ymin':45,'ymax':60}
fig,axes=plt.subplots(1,3,figsize=(18,6),gridspec_kw={'width_ratios':[1.1,.7,.7]})
for ax in axes:
    world_4326.plot(ax=ax,color='white',edgecolor='0.3',linewidth=.45)
    ax.set_xticks([]);ax.set_yticks([])
for area,points in gdf_4326.groupby('transboundary_area'):
    axes[0].scatter(points.geometry.x,points.geometry.y,s=13,label=AREA_SHORT.get(area,area),alpha=.8)
axes[0].legend(loc='lower left',fontsize=8,title='TBA')
for bounds,color in [(zoom_usmx,'tab:blue'),(zoom_europe,'tab:orange')]:
    axes[0].add_patch(Rectangle((bounds['xmin'],bounds['ymin']),
        bounds['xmax']-bounds['xmin'],bounds['ymax']-bounds['ymin'],
        fill=False,edgecolor=color,linewidth=1.8))
for ax,bounds,subset,title in [
    (axes[1],zoom_usmx,gdf_4326[gdf_4326.transboundary_area.eq('US-MX transboundary')],'US–MX'),
    (axes[2],zoom_europe,gdf_4326[~gdf_4326.transboundary_area.eq('US-MX transboundary')],'Europe')]:
    for area,points in subset.groupby('transboundary_area'):
        ax.scatter(points.geometry.x,points.geometry.y,s=18,label=AREA_SHORT.get(area,area),alpha=.85)
    ax.set_xlim(bounds['xmin'],bounds['xmax']);ax.set_ylim(bounds['ymin'],bounds['ymax'])
    ax.set_title(title); ax.legend(fontsize=7,loc='lower left')
axes[0].set_title('Study-area locations and zoom extents')
fig.tight_layout()
fig.savefig(FIG_DIR/'Figure_01_geographic_study_areas.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_01_geographic_study_areas.pdf',bbox_inches='tight')
plt.show()

### Figure 2 — Area-average monthly hydroclimatic series
Original figure is described as P, T, AET and PET; the supplied plot notebook additionally includes processed TWS. We preserve this **five-panel** plotted version, identifying TWS as the fifth panel.

In [ ]:
cols=['temp_era5','precip_era5','aet','pet','tws_processed']
series=(gdf_stats.assign(month=gdf_stats['date'].dt.to_period('M').dt.to_timestamp())
  .groupby(['transboundary_area','month'],as_index=False)[cols].mean())
fig,axes=plt.subplots(len(cols),1,figsize=(15,12),sharex=True)
for idx,(ax,col) in enumerate(zip(axes,cols)):
    for area in AREA_ORDER:
        s=series[series.transboundary_area.eq(area)]
        ax.plot(s.month,s[col],lw=1.2,label=AREA_SHORT[area])
    ax.set_ylabel(VARS_MAP[col]);ax.grid(alpha=.22)
    ax.text(.005,.92,f'{string.ascii_lowercase[idx]})',transform=ax.transAxes,weight='bold')
axes[-1].set_xlabel('Year');axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,-.025),ncol=4)
fig.tight_layout()
fig.savefig(FIG_DIR/'Figure_02_monthly_hydroclimatic_series.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_02_monthly_hydroclimatic_series.pdf',bbox_inches='tight')
plt.show()

### Figure 3 — Analytical design of the four-stage pipeline
The manuscript contains a **four-block conceptual diagram** but its source image/editable code was not included in the four notebooks. The following **new, code-generated diagram** reproduces the described stage ordering and inputs; it is *not pixel-identical* to the manuscript's embedded artwork.

In [ ]:
stages=[
 ('1. GRACE TWS processing','Processed/infilled TWS polygons\nWell-month TWS predictors'),
 ('2. Step-1 groundwater ML','P, T, AET, PET, TWS + temporal features\n4 candidate models; chronological CV'),
 ('3. Step-2 groundwater ML','96-month reference-well selection\nRelative reference GWL as extra feature'),
 ('4. Well-performance clustering','PCA + K-means + silhouette\nCramér V and OK/BAD residuals')]
fig,ax=plt.subplots(figsize=(14,6));ax.set_xlim(0,14);ax.set_ylim(0,6);ax.axis('off')
for i,(title,detail) in enumerate(stages):
    y=4.65-i*1.28
    rect=FancyBboxPatch((.8,y),12.4,.95,boxstyle='round,pad=0.06',
                        facecolor=['#dbe8f2','#e7f0f5','#deeeea','#ecf3dd'][i],edgecolor='0.3')
    ax.add_patch(rect);ax.text(1.05,y+.7,title,fontsize=13,weight='bold',va='center')
    ax.text(5.9,y+.55,detail,fontsize=10,va='center',ha='left')
    if i<3:ax.annotate('',xy=(7,y-.30),xytext=(7,y),arrowprops={'arrowstyle':'->','lw':1.8})
fig.tight_layout();fig.savefig(FIG_DIR/'Figure_03_analytical_pipeline.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_03_analytical_pipeline.pdf',bbox_inches='tight');plt.show()

### Figure 4 — Model-performance quality classes in Step 1 and Step 2
Replicates the manuscript's **three-panel class counts** for CV R², holdout R² and CV RMSE (two bars/area). Categories follow the thresholds illustrated by the manuscript: R² negative / 0–0.25 / 0.25–0.5 / 0.5–0.75 / >0.75; RMSE >2 / 1–2 / 0.5–1 / 0.25–0.5 / <0.25. Wells with missing metrics are not assigned to a class.

In [ ]:
def r2_class(s):
    return pd.cut(s,[-np.inf,0,.25,.5,.75,np.inf],right=False,
        labels=['negative','0–0.25','0.25–0.5','0.5–0.75','>0.75'])
def rmse_class(s):
    return pd.cut(s,[-np.inf,.25,.5,1,2,np.inf],right=False,
        labels=['<0.25','0.25–0.5','0.5–1','1–2','>2'])
def plot_performance_classes(cmp):
    metrics=[('CV R² mean','step1_CV_R2_mean','step2_CV_R2_mean',r2_class),
             ('Holdout R²','step1_TEST_R2','step2_TEST_R2',r2_class),
             ('CV RMSE mean','CV_RMSE mean','CV_RMSE mean_round2',rmse_class)]
    # Use merged table for RMSE and comparison table for the two R² metrics.
    fig,axes=plt.subplots(1,3,figsize=(19,6),sharey=True)
    colors=['#222222','#2e6eaa','#40aeb2','#8acb54','#c6e65c']
    tab=[]
    for ax,(title,c1,c2,clf) in zip(axes,metrics):
        src=merged if title.startswith('CV RMSE') else cmp
        class_order=list(clf(pd.Series([-.01,.1,.3,.6,.8]) if clf==r2_class
                        else pd.Series([.1,.3,.7,1.5,2.5])).cat.categories)
        if clf==rmse_class: class_order=class_order[::-1]  # >2 black; <0.25 green
        for ai,area in enumerate(AREA_ORDER):
            for si,(step,col) in enumerate([('R1',c1),('R2',c2)]):
                values=clf(pd.to_numeric(src.loc[src.transboundary_area.eq(area),col],errors='coerce'))
                counts=values.value_counts().reindex(class_order,fill_value=0)
                base=0
                for ci,cl in enumerate(class_order):
                    ax.bar(ai*2.6+si,counts.loc[cl],bottom=base,width=.72,
                           color=colors[ci],label=cl if ai==0 and si==0 else None)
                    base+=counts.loc[cl]
                    tab.append({'metric':title,'transboundary_area':area,'step':step,
                                'class':cl,'n_wells':int(counts.loc[cl])})
        ax.set_title(title);ax.set_xticks([i*2.6+.5 for i in range(4)])
        ax.set_xticklabels([AREA_SHORT[x] for x in AREA_ORDER]);ax.grid(axis='y',alpha=.2)
        for i in range(4):
            ax.text(i*2.6,-1.1,'R1',ha='center',fontsize=7)
            ax.text(i*2.6+1,-1.1,'R2',ha='center',fontsize=7)
    axes[0].set_ylabel('Number of wells')
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='lower center',ncol=5,bbox_to_anchor=(.5,-.03))
    fig.tight_layout()
    return fig,pd.DataFrame(tab)
fig,class_counts=plot_performance_classes(comparison_df)
class_counts.to_csv(TABLE_DIR/'18_figure4_performance_class_counts.csv',index=False)
fig.savefig(FIG_DIR/'Figure_04_performance_quality_classes.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_04_performance_quality_classes.pdf',bbox_inches='tight')
plt.show()

### Figure 5 — Reference-well measured and imputed relative GWL
Reference wells are identified from the **≥96-month Step-1 selection**, rather than from later Figure 6 example-well ranking. The original two-colour area plot separates timestamps with observed absolute GWL from times filled by the selected reference model. Plotting a predicted relative series at observed times does **not** make it an independent measured-series validation.

In [ ]:
fig,axes=plt.subplots(4,1,figsize=(15,12),sharex=False)
for i,area in enumerate(AREA_ORDER):
    ax=axes[i]
    if area not in set(best_well_by_area.transboundary_area): continue
    well=best_well_by_area.set_index('transboundary_area').loc[area,'well_ID']
    sub=gdf_ml_with_ref[(gdf_ml_with_ref.transboundary_area==area)&
                         (gdf_ml_with_ref.well_ID==well)].sort_values('date')
    s=sub['gw_level_rel_ref']; obs=s.where(sub['gw_level_abs'].notna(),0)
    imp=s.where(sub['gw_level_abs'].isna(),0)
    ax.stackplot(sub.date,obs,imp,labels=['Observed timestamp','Missing timestamp (modelled)'],alpha=.8)
    ax.set_title(f'{AREA_SHORT[area]} — reference well {well}',loc='left')
    ax.set_ylabel('Relative GWL, m');ax.grid(alpha=.2)
    ax.text(.01,.90,f'{string.ascii_lowercase[i]})',transform=ax.transAxes,weight='bold')
axes[0].legend(loc='upper right',fontsize=9)
fig.tight_layout();fig.savefig(FIG_DIR/'Figure_05_reference_well_series.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_05_reference_well_series.pdf',bbox_inches='tight');plt.show()

### Figure 6 — Four selected non-reference well holdout examples
Original **Figure 6 sample-selection rule recovered from the fourth notebook**, code cell 26: exclude reference wells; require predictions in both rounds and **more than 24 distinct test dates**; within each TBA pick the well with **highest Step-2 mean CV R²** (ties: more dates then lexicographically smaller well_ID). This is **not** selection by greatest Step-1→Step-2 improvement; the rationale and IDs are exported in `07_figure6_sample_selection.csv`.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(16,9),squeeze=False)
for idx,area in enumerate(AREA_ORDER):
    ax=axes.flat[idx];sel=sample_wells[sample_wells.transboundary_area==area]
    if sel.empty: ax.set_visible(False);continue
    well=sel.iloc[0]['well_ID']
    subset=obs_pred_df[(obs_pred_df.transboundary_area==area)&(obs_pred_df.well_ID==well)]
    observed=subset.groupby('date',as_index=False)['y_obs'].mean().sort_values('date')
    predicted=subset.pivot_table(index='date',columns='prediction_step',values='y_pred').reset_index()
    ax.plot(observed.date,observed.y_obs,label='Observed GWL',lw=2)
    if 'step1' in predicted:ax.plot(predicted.date,predicted.step1,'--',label='Step 1')
    if 'step2' in predicted:ax.plot(predicted.date,predicted.step2,':',label='Step 2')
    ax.set_title(f'{AREA_SHORT[area]} · {well} · {int(sel.iloc[0].n_dates)} holdout dates')
    ax.set_ylabel('Absolute GWL, m');ax.grid(alpha=.2);ax.legend(fontsize=9)
    ax.text(.01,.92,f'{string.ascii_lowercase[idx]})',transform=ax.transAxes,weight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR/'Figure_06_nonreference_holdout_examples.jpg',dpi=DPI,bbox_inches='tight')
fig.savefig(FIG_DIR/'Figure_06_nonreference_holdout_examples.pdf',bbox_inches='tight')
plt.show()

### Figure 7 — Step-2 SHAP beeswarm summary (four areas)
Figure preserves the source plotting notebook's **pooled set of wells with positive Step-2 mean CV R²** (top features by mean absolute SHAP, ten per area, random seed 42). The manuscript caption describes “best-performing models”; the actual source plot pools positive-performing winners, so it should not be described as the single best well per region without changing selection.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# SHAP beeswarm in ONE figure with 4 subplots (2x2)
# Common symmetric X-axis scale across all panels
# Built from flattened CSV exported by notebook 1
# =========================================================

well_col = "well_ID"
area_col = "transboundary_area"
perf_col = "CV_R2 mean_round2"
threshold = 0
top_n_features = 10
save_path = str(FIG_DIR / 'Figure_07_shap_beeswarm.jpg')
dpi = DPI

title_fontsize = 18
subplot_title_fontsize = 16
axis_label_fontsize = 14
tick_fontsize = 12
panel_letter_fontsize = 16
colorbar_label_fontsize = 16
colorbar_tick_fontsize = 15

point_size = 14
jitter_strength = 0.32
figsize = (16, 12)
max_points_per_feature = 3000

FEATURE_LABEL_MAP = {
    "aet": "Actual evapotranspiration",
    "pet": "Potential evapotranspiration",
    "precip_era5": "Precipitation",
    "temp_era5": "Air temperature",
    "temp_era5_lag1": "Air temperature (1-month lag)",
    "temp_era5_lag2": "Air temperature (2-month lag)",
    "temp_era5_lag3": "Air temperature (3-month lag)",
    "temp_era5_lag6": "Air temperature (6-month lag)",
    "temp_era5_lag9": "Air temperature (9-month lag)",
    "temp_era5_lag12": "Air temperature (12-month lag)",
    "precip_era5_lag1": "Precipitation (1-month lag)",
    "precip_era5_lag2": "Precipitation (2-month lag)",
    "precip_era5_lag3": "Precipitation (3-month lag)",
    "precip_era5_lag6": "Precipitation (6-month lag)",
    "precip_era5_lag9": "Precipitation (9-month lag)",
    "precip_era5_lag12": "Precipitation (12-month lag)",
    "aet_lag1": "Actual evapotranspiration (1-month lag)",
    "aet_lag2": "Actual evapotranspiration (2-month lag)",
    "aet_lag3": "Actual evapotranspiration (3-month lag)",
    "aet_lag6": "Actual evapotranspiration (6-month lag)",
    "aet_lag9": "Actual evapotranspiration (9-month lag)",
    "aet_lag12": "Actual evapotranspiration (12-month lag)",
    "pet_lag1": "Potential evapotranspiration (1-month lag)",
    "pet_lag2": "Potential evapotranspiration (2-month lag)",
    "pet_lag3": "Potential evapotranspiration (3-month lag)",
    "pet_lag6": "Potential evapotranspiration (6-month lag)",
    "pet_lag9": "Potential evapotranspiration (9-month lag)",
    "pet_lag12": "Potential evapotranspiration (12-month lag)",
    "temp_era5_roll3_mean": "Air temperature (3-month mean)",
    "temp_era5_roll6_mean": "Air temperature (6-month mean)",
    "temp_era5_roll9_mean": "Air temperature (9-month mean)",
    "precip_era5_roll3_mean": "Precipitation (3-month mean)",
    "precip_era5_roll6_mean": "Precipitation (6-month mean)",
    "precip_era5_roll9_mean": "Precipitation (9-month mean)",
    "aet_roll3_mean": "Actual evapotranspiration (3-month mean)",
    "aet_roll6_mean": "Actual evapotranspiration (6-month mean)",
    "aet_roll9_mean": "Actual evapotranspiration (9-month mean)",
    "pet_roll3_mean": "Potential evapotranspiration (3-month mean)",
    "pet_roll6_mean": "Potential evapotranspiration (6-month mean)",
    "pet_roll9_mean": "Potential evapotranspiration (9-month mean)",
    "p_roll3_sum": "Precipitation (3-month sum)",
    "p_roll6_sum": "Precipitation (6-month sum)",
    "p_roll9_sum": "Precipitation (9-month sum)",
    "month_sin": "Month seasonality (sin)",
    "month_cos": "Month seasonality (cos)",
    "tws_processed": "Terrestrial water storage",
    "tws_processed_lag1": "Terrestrial water storage (1-month lag)",
    "tws_processed_lag2": "Terrestrial water storage (2-month lag)",
    "tws_processed_lag3": "Terrestrial water storage (3-month lag)",
    "tws_processed_lag6": "Terrestrial water storage (6-month lag)",
    "tws_processed_lag9": "Terrestrial water storage (9-month lag)",
    "tws_processed_lag12": "Terrestrial water storage (12-month lag)",
    "tws_processed_roll3_mean": "Terrestrial water storage (3-month mean)",
    "tws_processed_roll6_mean": "Terrestrial water storage (6-month mean)",
    "tws_processed_roll9_mean": "Terrestrial water storage (9-month mean)",
    "gw_level_rel_ref": "Groundwater level anomaly",
}

AREA_LABEL_MAP = {
    "LT-LV transboundary": "LT–LV",
    "LV-EE transboundary": "LV–EE",
    "PL-UA transboundary": "PL–UA",
    "US-MX transboundary": "US–MX",
}


def humanize_feature_name(feature_name, label_map=FEATURE_LABEL_MAP):
    return label_map.get(feature_name, feature_name)


# =========================================================
# 1. Select wells with positive Step 2 performance
# =========================================================

perf_df = (
    per_well_baseline_vs_ml[[well_col, area_col]]
    .merge(
        merged[[well_col, perf_col]],
        on=well_col,
        how="left"
    )
)

perf_df = perf_df[perf_df[perf_col] > threshold].copy()

if perf_df.empty:
    raise ValueError(
        f"No wells found with {perf_col} > {threshold}"
    )

perf_df[well_col] = perf_df[well_col].astype(str)

valid_wells = set(perf_df[well_col].unique())

well_to_area = (
    perf_df
    .drop_duplicates(well_col)
    .set_index(well_col)[area_col]
    .to_dict()
)


# =========================================================
# 2. Prepare SHAP data
# =========================================================

shap_plot_df = shap_long_df.copy()
shap_plot_df[well_col] = shap_plot_df[well_col].astype(str)

shap_plot_df = shap_plot_df[
    shap_plot_df[well_col].isin(valid_wells)
].copy()

shap_plot_df[area_col] = (
    shap_plot_df[well_col].map(well_to_area)
)

shap_plot_df = shap_plot_df.dropna(
    subset=[
        area_col,
        "feature_name",
        "shap_value",
        "feature_value"
    ]
).copy()

shap_plot_df["shap_value"] = pd.to_numeric(
    shap_plot_df["shap_value"], errors="coerce"
)

shap_plot_df["feature_value"] = pd.to_numeric(
    shap_plot_df["feature_value"], errors="coerce"
)

shap_plot_df = shap_plot_df[
    np.isfinite(shap_plot_df["shap_value"]) &
    np.isfinite(shap_plot_df["feature_value"])
].copy()

shap_plot_df["feature_name_pretty"] = (
    shap_plot_df["feature_name"]
    .map(humanize_feature_name)
)

if shap_plot_df.empty:
    raise ValueError(
        "No usable SHAP rows found after filtering."
    )


# =========================================================
# 3. Identify top features in each area
# =========================================================

areas_sorted = sorted(
    shap_plot_df[area_col].dropna().unique().tolist()
)

if len(areas_sorted) != 4:
    print(
        f"Warning: expected 4 transboundary areas, "
        f"found {len(areas_sorted)}."
    )

feature_orders = {}

for area in areas_sorted:
    area_df = shap_plot_df[
        shap_plot_df[area_col] == area
    ]

    top_features = (
        area_df
        .groupby("feature_name_pretty")["shap_value"]
        .apply(lambda s: np.mean(np.abs(s.to_numpy())))
        .sort_values(ascending=False)
        .head(top_n_features)
        .index.tolist()
    )

    feature_orders[area] = top_features[::-1]


# =========================================================
# 4. Prepare points for plotting
#    Apply the same sampling logic before calculating
#    common X-axis limits
# =========================================================

rng = np.random.default_rng(42)

plot_data = {}
all_plotted_shap_values = []

for area in areas_sorted:

    area_df = shap_plot_df[
        shap_plot_df[area_col] == area
    ]

    plot_data[area] = []

    for yi, fname in enumerate(feature_orders[area]):

        d = area_df[
            area_df["feature_name_pretty"] == fname
        ]

        x = d["shap_value"].to_numpy(dtype=float)
        c = d["feature_value"].to_numpy(dtype=float)

        mask = np.isfinite(x) & np.isfinite(c)
        x = x[mask]
        c = c[mask]

        if len(x) == 0:
            continue

        if len(x) > max_points_per_feature:
            idx = rng.choice(
                len(x),
                size=max_points_per_feature,
                replace=False
            )
            x = x[idx]
            c = c[idx]

        c_lo, c_hi = (
            np.nanpercentile(c, [5, 95])
            if len(c) > 5
            else (np.nanmin(c), np.nanmax(c))
        )

        if c_lo == c_hi:
            c_hi = c_lo + 1e-9

        c_scaled = np.clip(
            (c - c_lo) / (c_hi - c_lo),
            0,
            1
        )

        y = yi + rng.uniform(
            -jitter_strength,
            jitter_strength,
            size=len(x)
        )

        plot_data[area].append({
            "x": x,
            "y": y,
            "c": c_scaled
        })

        all_plotted_shap_values.append(x)


# =========================================================
# 5. Calculate COMMON symmetric X-axis limits
#    Based only on plotted top-feature SHAP values
# =========================================================

if not all_plotted_shap_values:
    raise ValueError("No SHAP values available for plotting.")

all_x = np.concatenate(all_plotted_shap_values)

x_limit = np.max(np.abs(all_x))

if not np.isfinite(x_limit) or x_limit == 0:
    x_limit = 1.0

x_limit *= 1.05  # 5% padding

print(
    f"Common SHAP x-axis range: "
    f"{-x_limit:.3f} to {x_limit:.3f}"
)


# =========================================================
# 6. Draw beeswarm subplots
# =========================================================

def draw_beeswarm_subplot(
    ax,
    area,
    area_label,
    panel_letter
):

    feature_order = feature_orders[area]

    scatter_handle = None

    for item in plot_data[area]:

        scatter_handle = ax.scatter(
            item["x"],
            item["y"],
            c=item["c"],
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            s=point_size,
            alpha=0.8,
            linewidths=0
        )

    ax.axvline(
        0,
        color="gray",
        linewidth=1
    )

    # SAME X-AXIS SCALE FOR ALL SUBPLOTS
    ax.set_xlim(-x_limit, x_limit)

    ax.set_yticks(
        np.arange(len(feature_order))
    )

    ax.set_yticklabels(
        feature_order,
        fontsize=tick_fontsize
    )

    ax.tick_params(
        axis="x",
        labelsize=tick_fontsize,
        labelbottom=True
    )

    ax.set_xlabel(
        "SHAP value",
        fontsize=axis_label_fontsize
    )

    ax.set_title(
        area_label,
        fontsize=subplot_title_fontsize,
        pad=10
    )

    ax.grid(
        axis="x",
        alpha=0.15
    )

    ax.set_axisbelow(True)

    ax.text(
        0.01,
        0.02,
        f"{panel_letter})",
        transform=ax.transAxes,
        fontsize=panel_letter_fontsize,
        fontweight="bold",
        ha="left",
        va="bottom",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.85,
            pad=1.5
        )
    )

    return scatter_handle


# =========================================================
# 7. Create 2x2 figure
# =========================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=figsize,
    constrained_layout=True,
    sharex=True
)

axes = axes.ravel()

last_scatter = None

for i, ax in enumerate(axes):

    if i < len(areas_sorted):

        area = areas_sorted[i]

        area_label = AREA_LABEL_MAP.get(
            area,
            area
        )

        panel_letter = chr(ord("a") + i)

        last_scatter = draw_beeswarm_subplot(
            ax,
            area,
            area_label,
            panel_letter
        )

    else:
        ax.axis("off")


# =========================================================
# 8. Common colorbar
# =========================================================

if last_scatter is not None:

    cbar = fig.colorbar(
        last_scatter,
        ax=axes.tolist(),
        pad=0.01,
        shrink=0.9
    )

    cbar.set_label(
        "Feature value",
        fontsize=colorbar_label_fontsize
    )

    cbar.ax.tick_params(
        labelsize=colorbar_tick_fontsize
    )

    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Low", "High"])


# =========================================================
# 9. Save and display
# =========================================================

plt.savefig(
    save_path,
    dpi=dpi,
    bbox_inches="tight",
    format="jpeg"
)

plt.show()

### Figure 8 — Record length versus clustering performance
Mean Step-2 CV R² and mean number of train+holdout observations computed per **static-attribute cluster**. Each bubble indicates one cluster, with size proportional to well count. Interpretation is descriptive, not causal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# existing dataframe in memory
df = lean_all_areas_labeled.copy()

AREA_COL = "transboundary_area"
CLUSTER_COL = "cluster"
R2_COL = "CV_R2 mean_round2"

# make total record length
df["record_length_total"] = (
    pd.to_numeric(df["n_train"], errors="coerce").fillna(0) +
    pd.to_numeric(df["n_test"], errors="coerce").fillna(0)
)

df[CLUSTER_COL] = pd.to_numeric(df[CLUSTER_COL], errors="coerce")
df[R2_COL] = pd.to_numeric(df[R2_COL], errors="coerce")

df = df.dropna(subset=[AREA_COL, CLUSTER_COL, R2_COL, "record_length_total"]).copy()

# summarize per cluster within each area
cluster_summary = (
    df.groupby([AREA_COL, CLUSTER_COL], as_index=False)
      .agg(
          mean_record_length=("record_length_total", "mean"),
          mean_r2=(R2_COL, "mean"),
          n_wells=("well_ID", "count") if "well_ID" in df.columns else (R2_COL, "count")
      )
)

areas = cluster_summary[AREA_COL].unique().tolist()

# keep only first 4 areas for a 2x2 layout
areas = areas[:4]

# panel labels
panel_labels = ["a)", "b)", "c)", "d)"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(areas):
        ax.axis("off")
        continue

    area = areas[i]
    d = cluster_summary[cluster_summary[AREA_COL] == area].copy()

    ax.scatter(
        d["mean_record_length"],
        d["mean_r2"],
        s=50 + d["n_wells"] * 8,
        alpha=0.8
    )

    # cluster labels
    for _, row in d.iterrows():
        ax.annotate(
            f"C{int(row[CLUSTER_COL])}",
            (row["mean_record_length"], row["mean_r2"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            ha="left",
            va="bottom"
        )

    # subplot label
    ax.text(
        -0.12, 1.05,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        ha="right",
        va="bottom"
    )

    ax.set_title(area)
    ax.set_xlabel("Average record length")
    ax.set_ylabel("Average CV R²")
    ax.grid(True, alpha=0.3)

fig.savefig(str(FIG_DIR / 'Figure_08_record_length_vs_skill.jpg'), dpi=600, format="jpeg", bbox_inches="tight")
plt.show()

### Figure 9 — Cramér's V associations with K-means clusters
Rows are aquifer lithology, karst, confinement and pumping; columns are the four transboundary regions. Colour and printed value represent the strength of the descriptive cluster–category association.

In [ ]:
heat = (
    lean_all_areas_cat_assoc_cramersV.pivot(
        index="categorical_feature",
        columns="transboundary_area",
        values="cramers_v"
    )
    .reindex(index=["aquifer_lithology", "aquifer_karst", "confinement", "pumping"])
    .reindex(columns=AREA_ORDER)
)

# Human-readable labels
row_label_map = {
    "aquifer_lithology": "Aquifer lithology",
    "aquifer_karst": "Karst",
    "confinement": "Confinement",
    "pumping": "Pumping",
}

col_label_map = {
    "PL-UA transboundary": "PL–UA",
    "LT-LV transboundary": "LT–LV",
    "LV-EE transboundary": "LV–EE",
    "US-MX transboundary": "US–MX",
}

heat.index = [row_label_map.get(x, x) for x in heat.index]
heat.columns = [col_label_map.get(x, x) for x in heat.columns]

fig, ax = plt.subplots(figsize=(10, 5.8))

mat = heat.values.astype(float)

# Lighter color scheme
im = ax.imshow(mat, aspect="auto", vmin=0, vmax=1, cmap="RdBu_r", alpha=0.8)

ax.set_xticks(np.arange(len(heat.columns)))
ax.set_yticks(np.arange(len(heat.index)))
ax.set_xticklabels(heat.columns)
ax.set_yticklabels(heat.index)

plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
plt.setp(ax.get_yticklabels(), rotation=0)

# Annotate cells with black text
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat[i, j]
        label = "NA" if pd.isna(val) else f"{val:.2f}"
        ax.text(
            j, i, label,
            ha="center", va="center",
            fontsize=13,
            fontweight="bold",
            color="black"
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Cramér's V", fontsize=14)
cbar.ax.tick_params(labelsize=12)

# Cell borders
ax.set_xticks(np.arange(-0.5, len(heat.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heat.index), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

fig.subplots_adjust(left=0.23, right=0.92, bottom=0.16, top=0.96)
fig.savefig(str(FIG_DIR / 'Figure_09_cramersV_heatmap.jpg'), dpi=600, format="jpeg", bbox_inches="tight")
plt.show()

## Supplementary figures and reproducibility tables
Additional diagnostics from the original notebooks: missing-value availability by TBA, hydroclimate boxplots, k-selection silhouette profiles, standardized residuals; plus full well-level/per-area outputs. The manuscript DOCX contains **figure captions but no native Word tables**, so the notebook exports all substantively tabular results documented in the four source notebooks rather than inventing numbered manuscript tables.

In [ ]:
# Availability matrix (one figure per TBA), including missing monthly dates.
for area in AREA_ORDER:
    sub=gdf_stats[gdf_stats.transboundary_area.eq(area)].copy()
    if sub.empty: continue
    monthly=sub.assign(month=sub.date.dt.to_period('M'))
    avail=monthly.groupby(['well_ID','month'])['gw_level_abs'].apply(lambda s:s.notna().any())
    mat=avail.unstack('month')
    full_months=pd.period_range(mat.columns.min(),mat.columns.max(),freq='M')
    mat=mat.reindex(columns=full_months).fillna(False).astype(int)
    mat=mat.loc[mat.sum(axis=1).sort_values(ascending=False).index]
    fig,ax=plt.subplots(figsize=(13,max(4,min(18,len(mat)*.22))))
    ax.imshow(mat,aspect='auto',interpolation='nearest',cmap='Greys',vmin=0,vmax=1)
    ax.set_yticks(range(len(mat)));ax.set_yticklabels(mat.index,fontsize=6)
    ticks=np.arange(0,len(full_months),max(1,len(full_months)//12))
    ax.set_xticks(ticks);ax.set_xticklabels([str(full_months[j])[:7] for j in ticks],rotation=45,ha='right')
    ax.set_title(f'Appendix — measured GWL monthly availability, {AREA_SHORT[area]}')
    fig.tight_layout()
    fig.savefig(FIG_DIR/f'Appendix_Availability_{AREA_SHORT[area].replace("–","-")}.jpg',
                dpi=DPI,bbox_inches='tight');plt.close(fig)

# Hydroclimate boxplots across regions (not part of the 9 numbered figures).
fig,axes=plt.subplots(len(VARS_MAP),1,figsize=(12,13))
for ax,(col,label) in zip(axes,VARS_MAP.items()):
    ax.boxplot([gdf_stats.loc[gdf_stats.transboundary_area==area,col].dropna()
                for area in AREA_ORDER],tick_labels=[AREA_SHORT[a] for a in AREA_ORDER],vert=False)
    ax.set_title(label);ax.grid(axis='x',alpha=.2)
fig.tight_layout();fig.savefig(FIG_DIR/'Appendix_Hydroclimate_boxplots.jpg',dpi=DPI,bbox_inches='tight');plt.close(fig)

# Silhouette score profiles (supplementary original diagnostic).
if not lean_all_areas_silhouette_scores.empty:
    fig,axes=plt.subplots(2,2,figsize=(11,8))
    for ax,area in zip(axes.flat,AREA_ORDER):
        sub=lean_all_areas_silhouette_scores[
            lean_all_areas_silhouette_scores.transboundary_area==area].sort_values('k')
        if sub.empty: ax.set_visible(False);continue
        ax.plot(sub.k,sub.silhouette_score,'o-')
        ax.set_title(AREA_SHORT[area]);ax.set_xlabel('k');ax.set_ylabel('Silhouette score')
        ax.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR/'Appendix_Silhouette_profiles.jpg',dpi=DPI,bbox_inches='tight');plt.close(fig)

# Supplementary OK/BAD standardized-residual heatmap by TBA.
resid=lean_all_areas_okbad_contrib_stdresid.copy()
if not resid.empty:
    resid['level_label']=resid['categorical_feature'].astype(str)+': '+resid['level'].astype(str)
    fig,axes=plt.subplots(2,2,figsize=(15,11),squeeze=False)
    for ax,area in zip(axes.flat,AREA_ORDER):
        sub=resid[resid.transboundary_area==area]
        if sub.empty:ax.set_visible(False);continue
        top=sub.groupby('level_label')['abs_std_resid'].max().nlargest(12).index
        mat=sub[sub.level_label.isin(top)].pivot_table(
            index='level_label',columns='group',values='std_resid',aggfunc='mean')
        mat=mat.reindex(index=top)
        im=ax.imshow(mat.fillna(0),cmap='RdBu_r',vmin=-3,vmax=3,aspect='auto')
        ax.set_yticks(range(len(mat)));ax.set_yticklabels(mat.index,fontsize=7)
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels([str(x) for x in mat.columns])
        ax.set_title(AREA_SHORT[area])
    fig.colorbar(im,ax=axes.ravel().tolist(),label='Standardized residual',shrink=.7)
    fig.savefig(FIG_DIR/'Appendix_OK_BAD_standardized_residuals.jpg',dpi=DPI,bbox_inches='tight')
    plt.close(fig)

# Export a machine-readable run/software manifest.
try:
    import sklearn, scipy
    versions={'python':platform.python_version(),'pandas':pd.__version__,
              'geopandas':gpd.__version__,'numpy':np.__version__,
              'scikit_learn':sklearn.__version__,'shap':shap.__version__,
              'matplotlib':__import__('matplotlib').__version__,'scipy':scipy.__version__}
except Exception as exc: versions={'version_error':str(exc)}
(OUTPUT_DIR/'software_versions.json').write_text(json.dumps(versions,indent=2))
output_index=pd.DataFrame([{'path':str(p),'bytes':p.stat().st_size} for p in
    sorted(OUTPUT_DIR.rglob('*')) if p.is_file()])
output_index.to_csv(OUTPUT_DIR/'output_index.csv',index=False)
print('Exported',len(output_index),'artifacts to',OUTPUT_DIR.resolve())
display(output_index)

## Audit checks to report with the results
The manuscript states **67 / 140 wells** had increased mean CV R² (with TBA counts 30/55 LT–LV, 13/17 LV–EE, 19/56 PL–UA, 5/12 US–MX). The counts in the submitted files may shift with input-file revisions; **they are calculated above, not hard-coded**. Compare the generated `05_area_level_improvement.csv` against the manuscript before publication. If values differ, investigate source hashes, the exact reference-well exclusion set, duplicates, dependencies and sample windows.

**Scientific caveats requiring manuscript discussion:** fold-level imputer leakage, all-record reference fitting before Step-2 holdout, row-based vs calendar-month lagging where months are absent, overlapping GRACE polygon tie handling, apparent differences in manuscript Figure-4 caption numbering, Figure-6 eligibility/selection, and positive-well pooling in Figure 7. The notebook intentionally exposes these rather than presenting the output as a stricter reanalysis.